# 統計分析 中級編（Python）

このノートブックでは、Python を使った **統計分析** を、たくさんの例題を通して学びます。
記述統計から確率分布、推定、仮説検定、回帰分析、時系列の基礎までを扱います。
すべてのコードはブラウザ上（JupyterLite の Pyodide カーネル）で動くので、環境構築は不要です。

## このノートブックの使い方

- コードセルをクリックして **Shift + Enter** を押すと、そのセルが実行され、次のセルに移動します。
- **上から順番に** 実行してください。前のセルで作ったデータや関数を、後のセルで使うことがあります。
- **最初のセルはライブラリと日本語フォントの読み込みのため、実行に時間がかかります**（数十秒程度）。`[*]` の表示が数字に変わるまで待ってください。
- 乱数を使う例題は `np.random.default_rng(シード)` で乱数を固定しているので、何度実行しても同じ結果になります。シードを変えて結果がどう変わるか試してみるのもおすすめです。
- 途中で分からなくなったら、メニューの **Kernel → Restart Kernel and Run All Cells...** で最初からやり直せます。

## 前提知識

- Python の基本文法（変数、リスト、関数、for 文など）。不安な人は先に `python-tutorial.ipynb` を見てください。
- 高校数学程度の確率・統計（平均、分散、確率）。忘れていても例題の中で復習します。

## 使うライブラリ

| ライブラリ | 用途 |
|---|---|
| `numpy` | 数値計算、乱数 |
| `pandas` | 表形式データの集計 |
| `scipy.stats` | 確率分布、仮説検定、相関 |
| `statsmodels` | 回帰分析、分散分析の事後検定、時系列分解 |
| `matplotlib` | グラフ描画 |

## 目次

1. 記述統計（例題 1〜6）
2. データの可視化（例題 7〜13）
3. pandas によるデータ集計（例題 14〜17）
4. 確率分布（例題 18〜22）
5. 標本と推定（例題 23〜26）
6. 仮説検定（例題 27〜38）
7. 相関と回帰（例題 39〜46）
8. 時系列の基礎（例題 47〜49）
9. 練習問題（解答例付き）
10. まとめと次のステップ

## 例題の構成

各例題は **「問題設定 → コード → 結果の読み方」** の 3 つで構成されています。
コードを実行したら、出力された数字やグラフと「結果の読み方」を見比べながら読み進めてください。

In [ ]:
import piplite
await piplite.install("matplotlib-fontja==1.1.0")

import matplotlib_fontja
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

matplotlib_fontja.japanize()

表示の設定をしておきます（小数点以下 3 桁まで表示）。バージョンも確認しておきましょう。

In [ ]:
import scipy

pd.set_option("display.precision", 3)
np.set_printoptions(precision=3, suppress=True)
print(f"numpy {np.__version__} / pandas {pd.__version__} / scipy {scipy.__version__}")

## 1. 記述統計

記述統計とは、手元のデータの特徴を **数値やグラフで要約する** ことです。
「データの中心はどこか（代表値）」「どれくらい散らばっているか（散布度）」「形はどうか（歪み）」を調べます。

### 例題 1: 平均値・中央値・最頻値

**背景**: 10 人の生徒のテストの点数があります。このクラスの「代表的な点数」はいくつでしょうか。

**データ**: `62, 75, 81, 58, 90, 75, 68, 75, 99, 45`

3 種類の代表値を計算し、極端な値（外れ値）が加わったときにそれぞれどう変わるかを確かめます。

| 代表値 | 意味 |
|---|---|
| 平均値 (mean) | 合計 ÷ 個数 |
| 中央値 (median) | 小さい順に並べたときの真ん中の値 |
| 最頻値 (mode) | 最も多く出てくる値 |

In [ ]:
import statistics

scores = [62, 75, 81, 58, 90, 75, 68, 75, 99, 45]

# 標準ライブラリ statistics
print(f"平均値: {statistics.mean(scores):.2f}")
print(f"中央値: {statistics.median(scores):.2f}")
print(f"最頻値: {statistics.mode(scores)}")

# numpy / pandas でも同じことができる
arr = np.array(scores)
s = pd.Series(scores, name="score")
print(f"numpy  → 平均 {arr.mean():.2f}, 中央値 {np.median(arr):.2f}")
print(f"pandas → 平均 {s.mean():.2f}, 中央値 {s.median():.2f}, 最頻値 {s.mode().tolist()}")

# 外れ値（0 点の人）を 1 人追加すると…
scores_with_outlier = scores + [0]
print(f"\n0 点を追加 → 平均 {np.mean(scores_with_outlier):.2f}, 中央値 {np.median(scores_with_outlier):.2f}")

**結果の読み方**

- 平均値 72.8、中央値 75、最頻値 75 でした。分布がほぼ左右対称なら 3 つの値は近くなります。
- 0 点の人を 1 人加えると、平均は 66.2 まで下がりますが、中央値は 75 のまま動きません。中央値は **外れ値に強い（頑健な）** 代表値です。
- 年収や住宅価格のように極端に大きい値が混ざるデータでは、平均より中央値を使うことが多いです。「平均年収」が実感より高く感じるのはこのためです。
- 最頻値は、アンケートの選択肢のような **カテゴリデータ** でも使えます。`pd.Series.mode()` は最頻値が複数あるとすべて返します。

### 例題 2: 分散と標準偏差（標本 vs 母集団）

**背景**: 8 人の身長データから、ばらつきの大きさを数値で表したい。

**データ**: `158.2, 162.5, 170.1, 165.3, 174.8, 168.9, 160.4, 171.6`（cm）

分散は「平均からのずれの 2 乗」の平均、標準偏差はその平方根で、元のデータと同じ単位になります。
ここで注意すべきなのが、**n で割るか n−1 で割るか** です。

| 名前 | 割る数 | 使う場面 | numpy |
|---|---|---|---|
| 母分散 | n | 手元のデータが全体（母集団）そのもののとき | `np.var(x)` (ddof=0) |
| 不偏分散（標本分散） | n − 1 | 手元のデータは一部（標本）で、母集団のばらつきを推定したいとき | `np.var(x, ddof=1)` |

統計分析では多くの場合、手元のデータは標本なので **ddof=1** を使います。
ライブラリによってデフォルトが違うので、必ず確認しましょう。

In [ ]:
heights = [158.2, 162.5, 170.1, 165.3, 174.8, 168.9, 160.4, 171.6]
arr = np.array(heights)
n = len(arr)

var_pop = np.var(arr)             # ddof=0: n で割る（母分散）
var_sample = np.var(arr, ddof=1)  # ddof=1: n-1 で割る（不偏分散）
print(f"母分散   (n で割る)   : {var_pop:.3f}")
print(f"不偏分散 (n-1 で割る) : {var_sample:.3f}")
print(f"母標準偏差   : {np.sqrt(var_pop):.3f} cm")
print(f"標本標準偏差 : {np.sqrt(var_sample):.3f} cm")

print("\n--- ライブラリごとのデフォルトの違い ---")
print(f"numpy      np.std(x)        → {np.std(arr):.3f}  (ddof=0)")
print(f"pandas     Series.std()     → {pd.Series(arr).std():.3f}  (ddof=1)")
print(f"statistics pstdev / stdev   → {statistics.pstdev(heights):.3f} / {statistics.stdev(heights):.3f}")

# 手計算で確認
mean = arr.mean()
ss = ((arr - mean) ** 2).sum()
print(f"\n手計算: 平均 = {mean:.3f}, 偏差平方和 = {ss:.3f}, n = {n}")
print(f"  ss / n = {ss / n:.3f},  ss / (n - 1) = {ss / (n - 1):.3f}")

**結果の読み方**

- 標本標準偏差は約 5.9 cm。「身長は平均 166.5 cm を中心に、だいたい ±6 cm の範囲に散らばっている」と読めます。
- n で割るか n−1 で割るかで値が少し変わります。n が大きいほど差は小さくなりますが、小さい標本では無視できません。
- `np.std` は ddof=0、`pandas` の `.std()` は ddof=1 がデフォルトです。**同じ「標準偏差」でも計算結果が違う** ので、レポートに書くときはどちらを使ったかを明記しましょう。
- このノートブックでは以降、特に断りがなければ標本標準偏差（ddof=1）を使います。

### 例題 3: 四分位数・四分位範囲（IQR）・範囲と `describe()`

**背景**: 20 人の 1 か月の支出額（千円）から、データの散らばりを「位置」で表したい。

**データ**: `12, 15, 18, 20, 22, 22, 25, 27, 28, 30, 31, 33, 35, 38, 40, 42, 45, 50, 60, 85`

- 第 1 四分位数 Q1: 小さい方から 25% の位置の値
- 第 2 四分位数 Q2: 50%（中央値）
- 第 3 四分位数 Q3: 75%
- 四分位範囲 IQR = Q3 − Q1: 真ん中 50% のデータが収まる幅
- 範囲 = 最大値 − 最小値

pandas の `describe()` はこれらをまとめて計算してくれます。

In [ ]:
spending = [12, 15, 18, 20, 22, 22, 25, 27, 28, 30, 31, 33, 35, 38, 40, 42, 45, 50, 60, 85]
s = pd.Series(spending, name="月間支出(千円)")

q1, q2, q3 = np.percentile(spending, [25, 50, 75])
iqr = q3 - q1
print(f"第 1 四分位数 Q1        = {q1:.2f}")
print(f"第 2 四分位数 Q2 (中央値) = {q2:.2f}")
print(f"第 3 四分位数 Q3        = {q3:.2f}")
print(f"四分位範囲 IQR = Q3 - Q1 = {iqr:.2f}   (scipy.stats.iqr → {stats.iqr(spending):.2f})")
print(f"範囲 = 最大 - 最小       = {max(spending) - min(spending)}")
print(f"pandas の quantile: {s.quantile([0.25, 0.5, 0.75]).tolist()}")

# describe() でまとめて確認
s.describe()

**結果の読み方**

- Q1 = 22、中央値 = 30.5、Q3 = 40.5、IQR = 18.5。「真ん中半分の人は 22〜40.5 千円の間に収まっている」と読めます。
- 範囲は 73 ですが、これは最大値 85 という 1 人に引っ張られています。範囲は外れ値の影響を強く受けるため、IQR の方が散らばりの指標としては安定しています。
- `describe()` の出力は、`count`（個数）、`mean`、`std`（ddof=1 の標準偏差）、`min`、`25%`、`50%`、`75%`、`max` です。データを受け取ったらまず `describe()` を見る習慣をつけましょう。
- 四分位数の計算方法には複数の流儀があります（`np.percentile` の `method` 引数）。教科書と値が微妙に違っても慌てないでください。

### 例題 4: 歪度と尖度（分布の形）

**背景**: 「左右対称なデータ」と「右に裾が長いデータ（年収のようなデータ）」を比べて、分布の形を数値で表したい。

- **歪度 (skewness)**: 分布の非対称性。0 なら左右対称、正なら右に裾が長い、負なら左に裾が長い。
- **尖度 (kurtosis)**: 裾の重さ。scipy のデフォルトは正規分布を 0 とする定義（excess kurtosis）で、正なら正規分布より裾が重く、外れ値が出やすい。

ここでは乱数で 2 種類のデータを 1000 個ずつ作って比較します。

In [ ]:
rng = np.random.default_rng(4)
normal_data = rng.normal(loc=50, scale=10, size=1000)      # 左右対称なデータ
skewed_data = rng.lognormal(mean=3, sigma=0.6, size=1000)  # 右に裾が長いデータ（対数正規分布）

for name, x in [("正規分布　　", normal_data), ("対数正規分布", skewed_data)]:
    print(f"{name}: 平均 {x.mean():6.2f}, 中央値 {np.median(x):6.2f}, "
          f"歪度 {stats.skew(x):6.3f}, 尖度 {stats.kurtosis(x):6.3f}")

print(f"\npandas の .skew() / .kurt() は不偏推定量なので値がわずかに異なる: "
      f"歪度 {pd.Series(skewed_data).skew():.3f}, 尖度 {pd.Series(skewed_data).kurt():.3f}")

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
axes[0].hist(normal_data, bins=30, color="steelblue", edgecolor="white")
axes[0].set_title("左右対称な分布（歪度 ≈ 0）")
axes[1].hist(skewed_data, bins=30, color="darkorange", edgecolor="white")
axes[1].set_title("右に裾が長い分布（歪度 > 0）")
for ax in axes:
    ax.set_xlabel("値")
    ax.set_ylabel("度数")
plt.tight_layout()
plt.show()

**結果の読み方**

- 正規分布のデータは歪度・尖度ともに 0 に近く、平均と中央値もほぼ一致します。
- 対数正規分布のデータは歪度が約 2 と大きな正の値で、**平均が中央値より大きく** なっています。右側の長い裾（少数の大きな値）が平均を引き上げているからです。
- 尖度も大きな正の値です。裾が重い分布では「平均 ± 2SD の外側」に思った以上に多くのデータが出ます。
- 歪度の絶対値が 1 を超えるようなら、平均・標準偏差だけで要約するのは不十分です。中央値や四分位数を併用するか、対数変換（`np.log`）を検討しましょう。

### 例題 5: 外れ値の検出（IQR ルールと z スコア）

**背景**: 製品 50 個の計測値の中に、明らかにおかしい値（外れ値）が混ざっていそうです。機械的に検出したい。

代表的な 2 つのルールを使います。

| 方法 | 外れ値の基準 | 特徴 |
|---|---|---|
| IQR ルール | Q1 − 1.5×IQR 未満、または Q3 + 1.5×IQR 超（箱ひげ図のひげの外） | 外れ値自身の影響を受けにくい |
| z スコア | \|z\| = \|(x − 平均) / 標準偏差\| > 3（または 2） | 外れ値が平均・標準偏差を引っ張るので見逃しやすい |

乱数で正常なデータを作り、3 個の外れ値をわざと混ぜて試します。

In [ ]:
rng = np.random.default_rng(5)
values = rng.normal(loc=100, scale=15, size=50).round(1)
values[[3, 17, 42]] = [158.0, 38.0, 172.0]   # 3 つの外れ値を混ぜる
df = pd.DataFrame({"value": values})

# 方法 1: IQR ルール
q1, q3 = df["value"].quantile([0.25, 0.75])
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
df["iqr_outlier"] = (df["value"] < lower) | (df["value"] > upper)

# 方法 2: z スコア（|z| > 3）
df["z"] = stats.zscore(df["value"], ddof=1)
df["z_outlier"] = df["z"].abs() > 3

print(f"IQR ルールの正常範囲: {lower:.1f} 〜 {upper:.1f}")
print(f"IQR ルールで検出: {df['iqr_outlier'].sum()} 個,  z スコア (|z|>3) で検出: {df['z_outlier'].sum()} 個")
print(f"外れ値を除いた平均: {df.loc[~df['iqr_outlier'], 'value'].mean():.2f}  (除く前: {df['value'].mean():.2f})")

df[df["iqr_outlier"] | df["z_outlier"]]

**結果の読み方**

- IQR ルールは混ぜた 3 個の外れ値をすべて検出しました。一方、z スコア（|z| > 3）で検出できたのは 172.0 の 1 個だけで、158.0 と 38.0 は z ≈ 2.98, −2.91 とわずかに 3 に届きません。外れ値そのものが標準偏差を大きくしてしまい、z の値が 3 を超えにくくなるためです（**マスキング効果**）。
- 外れ値を機械的に削除してはいけません。入力ミスなのか、本当に起きた珍しい現象なのかを **必ず確認** し、削除した場合はその理由を記録しましょう。
- 外れ値の影響を受けにくい代表値（中央値）や、頑健な手法（後述のノンパラメトリック検定）を使うのも一つの対処法です。

### 例題 6: 標準化（z スコア）と偏差値

**背景**: 佐藤さんは数学も英語も 80 点でした。しかし数学は平均 69.5 点、英語は平均 79.5 点です。「どちらの科目の方が相対的に良い成績か」を比べたい。

単位や平均が違うデータを比べるには、**標準化**（平均 0、標準偏差 1 に変換）します。

$$z = \frac{x - \bar{x}}{s}$$

日本でおなじみの **偏差値** は、z を平均 50・標準偏差 10 に変換したものです: 偏差値 = 50 + 10z

In [ ]:
exam = pd.DataFrame({
    "name": ["佐藤", "鈴木", "高橋", "田中", "伊藤", "渡辺", "山本", "中村"],
    "math":    [80, 45, 92, 60, 71, 55, 88, 65],
    "english": [80, 78, 70, 92, 82, 68, 90, 76],
})

for subj in ["math", "english"]:
    mean = exam[subj].mean()
    std = exam[subj].std(ddof=0)   # 偏差値の計算では慣習的に母標準偏差 (ddof=0) を使う
    exam[f"{subj}_z"] = (exam[subj] - mean) / std
    exam[f"{subj}_偏差値"] = 50 + 10 * exam[f"{subj}_z"]
    print(f"{subj:8s}: 平均 {mean:.2f}, 標準偏差 {std:.2f}")

# scipy でも同じ計算ができる（ddof=0 がデフォルト）
print("scipy.stats.zscore(math) →", stats.zscore(exam["math"]).round(2).tolist())

print("\n佐藤さんは数学も英語も 80 点ですが…")
exam.round(2)

**結果の読み方**

- 佐藤さんの数学 80 点は z ≈ 0.68（偏差値 ≈ 56.8）、英語 80 点は z ≈ 0.06（偏差値 ≈ 50.6）。同じ 80 点でも **数学の方が相対的に良い成績** です。
- 標準化は「平均からの距離を標準偏差の何倍か」で測るので、科目・単位・満点が違っても比較できます。回帰分析や機械学習の前処理でもよく使います。
- 偏差値 60 は上位約 16%、70 は上位約 2.3% に相当します（データが正規分布に近い場合。例題 20 で確認します）。
- 標準化しても分布の **形は変わりません**。歪んだデータは標準化しても歪んだままです。

## 2. データの可視化

数値の要約だけでは分布の形や外れ値、グループ間の違いを見落とします。**分析の最初と最後には必ずグラフを描く** のが鉄則です。

### 例題 7: ヒストグラムと bins の影響

**背景**: 2 つの製造ラインで作られた製品の重量データ（合計 500 個）があります。分布の形を見たい。

ヒストグラムは区間（bin）の数によって見え方が大きく変わります。同じデータを bins = 4, 20, 100 で描き比べてみましょう。

In [ ]:
rng = np.random.default_rng(7)
# 2 つの山を持つデータ（ライン 1: 平均 48 g、ライン 2: 平均 54 g）
weights = np.concatenate([rng.normal(48, 1.5, 300), rng.normal(54, 1.5, 200)])

fig, axes = plt.subplots(1, 3, figsize=(10, 3.2))
for ax, bins in zip(axes, [4, 20, 100]):
    ax.hist(weights, bins=bins, color="steelblue", edgecolor="white")
    ax.set_title(f"bins = {bins}")
    ax.set_xlabel("重量 (g)")
axes[0].set_ylabel("度数")
plt.tight_layout()
plt.show()

n = len(weights)
print(f"n = {n}")
print(f"スタージェスの公式 1 + log2(n) → 約 {int(np.ceil(1 + np.log2(n)))} 区間")
print(f"numpy の bins='auto'          → {len(np.histogram_bin_edges(weights, bins='auto')) - 1} 区間")

**結果の読み方**

- bins = 4 では山が 1 つに見えてしまい、2 つのラインの違いが分かりません。bins = 100 では細かすぎてギザギザになり、形が読み取りにくくなります。bins = 20 程度でちょうど 2 つの山（**二峰性**）が見えます。
- bins の目安には「スタージェスの公式」や numpy の `bins="auto"` が使えますが、**複数の bins を試して形が安定しているか確認する** のが実践的です。
- 二峰性が見えたら「異なるグループが混ざっていないか」を疑いましょう。グループごとに分けて分析すると、それぞれは単峰の分布になることがよくあります。

### 例題 8: データの読み込みとカテゴリの確認（データクリーニング）

**背景**: ここからは、あやめ（iris）のデータセット `data/iris.csv` を使います。3 種（setosa, versicolor, virginica）の花のサイズが 50 個ずつ記録されている……はずです。分析を始める前に、本当にそうなっているかを確認します。

カテゴリ列を含むデータを受け取ったら、集計や検定を始める前に必ず `value_counts()` で **ラベルの種類と件数** を確認しましょう。表記ゆれ（`Setosa` と `setosa`）、余分な空白、入力途中で切れた値などが混ざっていると、グループ集計の結果が静かに狂います。

In [ ]:
iris = pd.read_csv("data/iris.csv")
print(iris.shape)
print(iris.head())

print("\n修正前の品種ごとの件数:")
print(iris["species"].value_counts())

# 1 件だけ "se" という途中で切れたラベルがある → "setosa" に修正する
iris["species"] = iris["species"].replace("se", "setosa")

print("\n修正後の品種ごとの件数:")
print(iris["species"].value_counts())

**結果の読み方**

- `head()` を見ると 1 行目の species が `se` と途中で切れています。`value_counts()` でも、setosa が 49 件しかなく、`se` という謎のカテゴリが 1 件あることが分かります。
- この行の数値（5.1, 3.5, 1.4, 0.2）は典型的な setosa の値なので、ここでは入力ミスと判断して `replace` で修正しました。実務では、元の記録や担当者に確認したうえで、**何をどう直したかを必ず記録に残します**。勝手に行を削除するのは最後の手段です。
- この確認を怠って `groupby("species")` すると、1 件だけの「se」という第 4 のグループができ、その平均が表に紛れ込み、標準偏差は NaN になります（1 件では計算できないため）。気づかないままレポートに載ってしまう典型的な事故です。
- 以降の例題では、この修正済みの `iris` を使います。

### 例題 9: 箱ひげ図でグループを比較する

**背景**: iris の 3 種で、花弁の長さ (petal_length) がどう違うかを比べたい。

箱ひげ図は、箱が Q1〜Q3（IQR）、箱の中の線が中央値、ひげが 1.5×IQR の範囲、その外側の点が外れ値を表します。

In [ ]:
species_list = ["setosa", "versicolor", "virginica"]
groups = [iris.loc[iris["species"] == sp, "petal_length"] for sp in species_list]

plt.figure(figsize=(6, 4))
plt.boxplot(groups)
plt.xticks([1, 2, 3], species_list)
plt.ylabel("花弁の長さ (cm)")
plt.title("種ごとの花弁の長さ（箱ひげ図）")
plt.show()

iris.groupby("species")["petal_length"].describe()

**結果の読み方**

- setosa の花弁は明らかに短く（中央値 1.5 cm）、他の 2 種と箱が全く重なりません。versicolor（中央値 4.35 cm）と virginica（中央値 5.55 cm）は一部重なっています。
- 箱の高さ（IQR）は種ごとのばらつきを表し、setosa は非常にばらつきが小さいことが分かります。
- 箱ひげ図はグループが多いときの比較に向いています。一方で、二峰性のような分布の形は箱ひげ図では分かりません。必要ならヒストグラムも併用してください。
- `describe()` の結果と箱ひげ図の各要素（25%、50%、75%）が対応していることを確認しましょう。

### 例題 10: 散布図で 2 変数の関係を見る

**背景**: iris の花弁の長さと幅の間に関係はあるでしょうか。種ごとに色分けした散布図を描きます。

In [ ]:
colors = {"setosa": "tab:blue", "versicolor": "tab:orange", "virginica": "tab:green"}

plt.figure(figsize=(6, 4))
for sp, color in colors.items():
    sub = iris[iris["species"] == sp]
    plt.scatter(sub["petal_length"], sub["petal_width"], s=18, color=color, label=sp, alpha=0.8)
plt.xlabel("花弁の長さ (cm)")
plt.ylabel("花弁の幅 (cm)")
plt.title("花弁の長さと幅の散布図")
plt.legend()
plt.show()

print(f"全体の相関係数: {iris['petal_length'].corr(iris['petal_width']):.3f}")
for sp in colors:
    sub = iris[iris["species"] == sp]
    print(f"  {sp:10s}: {sub['petal_length'].corr(sub['petal_width']):.3f}")

**結果の読み方**

- 花弁の長さと幅には強い正の関係があり、全体の相関係数は約 0.96 です。
- 種ごとに見ると点のかたまりがはっきり分かれています。**色分けすることで「グループの違い」と「グループ内の関係」を同時に見る** ことができます。
- 種ごとの相関係数は全体より小さくなります（setosa は約 0.31）。全体の強い相関の多くは「種による違い」で説明されています。このような構造は例題 45 でもう一度扱います。

### 例題 11: 複数系列の折れ線グラフ

**背景**: ある店の 3 商品の月次売上（万円）を 1 年分並べて、季節による違いを比べたい。時間の流れに沿ったデータは折れ線グラフが基本です。

In [ ]:
months = np.arange(1, 13)
monthly_sales = pd.DataFrame({
    "月": months,
    "アイスクリーム": [20, 22, 30, 45, 60, 85, 110, 120, 80, 50, 30, 25],
    "ホットコーヒー": [90, 85, 75, 60, 50, 40, 35, 35, 50, 65, 80, 95],
    "ミネラルウォーター": [40, 42, 45, 50, 58, 70, 85, 90, 70, 55, 45, 42],
}).set_index("月")

plt.figure(figsize=(6.5, 4))
for col in monthly_sales.columns:
    plt.plot(monthly_sales.index, monthly_sales[col], marker="o", label=col)
plt.xticks(months)
plt.xlabel("月")
plt.ylabel("売上 (万円)")
plt.title("商品別の月次売上")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

monthly_sales.agg(["sum", "mean", "max", "idxmax"]).T

**結果の読み方**

- アイスクリームとミネラルウォーターは夏（7〜8 月）にピーク、ホットコーヒーは冬にピークで、逆の動きをしています。
- `agg(["sum", "mean", "max", "idxmax"])` で商品ごとの合計・平均・最大値・最大の月を一度に求めています。`idxmax` は最大値の **インデックス（ここでは月）** を返します。
- 折れ線グラフはデータに順序（時間）があるときに使います。順序のないカテゴリ（例: 店舗名）を折れ線でつなぐのは誤解を招くのでやめましょう。

### 例題 12: groupby 集計と誤差棒付き棒グラフ

**背景**: iris の種ごとにがく片の長さ (sepal_length) の平均を棒グラフで比べたい。ばらつきの情報も誤差棒（エラーバー）で示します。

In [ ]:
summary = iris.groupby("species")["sepal_length"].agg(["mean", "std", "count"])
summary["se"] = summary["std"] / np.sqrt(summary["count"])   # 標準誤差（例題 24 参照）

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
labels = list(summary.index)
axes[0].bar(labels, summary["mean"], yerr=summary["std"], capsize=6, color=["tab:blue", "tab:orange", "tab:green"], alpha=0.8)
axes[0].set_title("誤差棒 = 標準偏差（データの散らばり）")
axes[1].bar(labels, summary["mean"], yerr=summary["se"], capsize=6, color=["tab:blue", "tab:orange", "tab:green"], alpha=0.8)
axes[1].set_title("誤差棒 = 標準誤差（平均の不確かさ）")
for ax in axes:
    ax.set_ylabel("がく片の長さの平均 (cm)")
plt.tight_layout()
plt.show()

summary

**結果の読み方**

- 平均は setosa < versicolor < virginica の順です。
- 誤差棒に **標準偏差** を使うと「個々のデータがどれくらい散らばっているか」、**標準誤差** を使うと「平均値がどれくらい正確に推定できているか」を表します。意味が全く違うので、グラフには必ずどちらを使ったかを明記してください。
- 標準誤差は標準偏差 ÷ √n なので、n が大きいほど小さくなります（50 個なので約 1/7）。
- 棒グラフは「0 から始まる軸」で描くのが原則です。軸の途中から始めると差が誇張されます。

### 例題 13: 相関行列をヒートマップで表示する

**背景**: iris の 4 つの数値変数について、すべてのペアの相関係数を一度に眺めたい。

`df.corr()` で相関行列を計算し、`plt.imshow` で色付きの表（ヒートマップ）にします。

In [ ]:
numeric = iris.select_dtypes("number")
corr = numeric.corr()

plt.figure(figsize=(5.5, 4.5))
plt.imshow(corr.values, cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar(label="相関係数")
plt.xticks(range(len(corr)), corr.columns, rotation=45, ha="right")
plt.yticks(range(len(corr)), corr.columns)
for i in range(len(corr)):
    for j in range(len(corr)):
        plt.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", color="black")
plt.title("iris の相関行列")
plt.tight_layout()
plt.show()

corr

**結果の読み方**

- 対角線は自分自身との相関なので常に 1 です。行列は対角線をはさんで対称です。
- petal_length と petal_width（0.96）、petal_length と sepal_length（0.87）は強い正の相関。sepal_width は他の変数と弱い負の相関です。
- 相関係数の目安: |r| < 0.2 ほぼ相関なし、0.2〜0.4 弱い、0.4〜0.7 中程度、0.7 以上 強い（分野によって基準は変わります）。
- 相関行列は **直線的な関係** しか捉えません。曲線的な関係や外れ値の影響は散布図で確認する必要があります（例題 40）。
- `corr()` は数値列にしか使えないので、`select_dtypes("number")` で文字列の列（species）を除いています。

## 3. pandas によるデータ集計

実務のデータ分析では「グループごとに集計する」作業が大半を占めます。pandas の `groupby`、`pivot_table`、`crosstab` を使いこなしましょう。

### 例題 14: groupby と agg（店舗別の売上集計）

**背景**: 3 店舗の 2024 年上半期の売上明細（300 件）があります。店舗ごとの件数・売上合計・平均単価・最大金額を求めたい。

データは乱数で作ります。`groupby("列名")` でグループに分け、`agg` で複数の統計量を一度に計算します。
`agg(新しい列名=("元の列", "関数名"))` の書き方（名前付き集計）を使うと、結果の列名を自由に決められます。

In [ ]:
rng = np.random.default_rng(13)
n = 300
sales = pd.DataFrame({
    "date": rng.choice(pd.date_range("2024-01-01", "2024-06-30"), size=n),
    "store": rng.choice(["東京", "大阪", "名古屋"], size=n, p=[0.5, 0.3, 0.2]),
    "category": rng.choice(["食品", "日用品", "衣料"], size=n),
    "quantity": rng.integers(1, 6, size=n),
})
unit_price = {"食品": 800, "日用品": 1200, "衣料": 3500}
sales["amount"] = (sales["quantity"] * sales["category"].map(unit_price) * rng.uniform(0.8, 1.2, size=n)).round(0)
sales = sales.sort_values("date").reset_index(drop=True)
print(sales.head())

print("\n店舗ごとの集計:")
by_store = sales.groupby("store").agg(
    件数=("amount", "size"),
    売上合計=("amount", "sum"),
    平均単価=("amount", "mean"),
    最大金額=("amount", "max"),
)
print(by_store.sort_values("売上合計", ascending=False))

print("\n店舗 × カテゴリごとの売上合計:")
sales.groupby(["store", "category"])["amount"].sum().unstack()

**結果の読み方**

- 東京は件数が最も多く（全体の約半分になるように乱数を設定しました）、売上合計も最大です。一方、平均単価は名古屋が最も高くなっています。ただし名古屋は件数が少ないので、この差が偶然なのかどうかは検定（例題 28 や 35）で確かめる必要があります。
- 2 つのキーで `groupby` すると結果は階層インデックス（MultiIndex）になります。`unstack()` で 2 つ目のキーを列に展開すると表として読みやすくなります。
- よく使う集計関数: `size`（行数）、`count`（欠損を除いた個数）、`sum`、`mean`、`median`、`std`、`min`、`max`、`nunique`（種類数）。

### 例題 15: pivot_table（店舗 × カテゴリ、月 × 店舗）

**背景**: 例題 14 の売上を「行 = 店舗、列 = カテゴリ」の表にまとめ、合計行・合計列も付けたい。さらに月別・店舗別の推移を棒グラフで見たい。

`pd.pivot_table(データ, values=集計する列, index=行, columns=列, aggfunc=関数)` は Excel のピボットテーブルと同じ考え方です。

In [ ]:
pivot = pd.pivot_table(sales, values="amount", index="store", columns="category",
                       aggfunc="sum", margins=True, margins_name="合計")
print("店舗 × カテゴリの売上合計:")
print(pivot.astype(int))

sales["month"] = sales["date"].dt.month
monthly = pd.pivot_table(sales, values="amount", index="month", columns="store", aggfunc="sum")

ax = monthly.plot(kind="bar", figsize=(6.5, 3.8), width=0.8)
ax.set_xlabel("月")
ax.set_ylabel("売上合計 (円)")
ax.set_title("月別・店舗別の売上")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

monthly.astype(int)

**結果の読み方**

- `margins=True` を付けると合計行・合計列が追加されます。行合計は店舗ごとの売上、列合計はカテゴリごとの売上です。
- `aggfunc` を `"mean"` にすれば平均単価の表、`"size"` にすれば件数の表になります。複数渡す（`aggfunc=["sum", "mean"]`）こともできます。
- DataFrame の `.plot(kind="bar")` を使うと、列ごとに色分けされた棒グラフを 1 行で描けます。
- 乱数で作ったデータなので月ごとの変動に意味はありません。実データで「差がある」と言うには、この後で学ぶ検定が必要です。

### 例題 16: クロス集計（crosstab）

**背景**: 200 人に「サービスへの満足度（満足 / 普通 / 不満）」を尋ね、年代も記録しました。年代によって満足度の分布が違うかを表にしたい。

2 つのカテゴリ変数の組み合わせの度数を数える表を **クロス集計表（分割表）** と言い、`pd.crosstab` で作れます。

In [ ]:
rng = np.random.default_rng(15)
n = 200
age = rng.choice(["20代", "30代", "40代", "50代以上"], size=n, p=[0.3, 0.3, 0.25, 0.15])
prob = {"20代": [0.6, 0.3, 0.1], "30代": [0.5, 0.35, 0.15],
        "40代": [0.35, 0.4, 0.25], "50代以上": [0.3, 0.4, 0.3]}
satisfaction = np.array([rng.choice(["満足", "普通", "不満"], p=prob[a]) for a in age])
survey = pd.DataFrame({"age": age, "satisfaction": satisfaction})

order = ["満足", "普通", "不満"]
ct = pd.crosstab(survey["age"], survey["satisfaction"], margins=True, margins_name="合計")
ct = ct[order + ["合計"]]          # 列の順番を並べ替える
print("度数のクロス集計表:")
print(ct)

print("\n行方向の割合（各年代の中での満足度の内訳）:")
ratio = pd.crosstab(survey["age"], survey["satisfaction"], normalize="index")[order]
print(ratio.round(3))

ax = ratio.plot(kind="barh", stacked=True, figsize=(6.5, 3.2), color=["tab:blue", "lightgray", "tab:red"])
ax.set_xlabel("割合")
ax.set_ylabel("年代")
ax.set_title("年代別の満足度の内訳")
ax.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()

**結果の読み方**

- 度数の表からは「20 代は満足が多く、50 代以上は不満の割合が高い」傾向が見えます。
- 年代ごとの人数が違うので、**割合に直して比べる** ことが重要です。`normalize="index"` で行ごとの割合、`"columns"` で列ごとの割合、`"all"` で全体に対する割合になります。
- 「年代と満足度に関連があるか」を統計的に判断するには、例題 34 のカイ二乗検定（独立性の検定）を使います。この表はそのままその検定の入力になります。

### 例題 17: iris データの種ごとの要約

**背景**: iris の 4 つの変数について、種ごとの平均と標準偏差を一覧表にしたい。複数の列 × 複数の統計量を一度に集計します。

In [ ]:
print("種ごとの件数:")
print(iris["species"].value_counts())

summary = iris.groupby("species").agg(["mean", "std"])
summary.columns = [f"{col}_{stat}" for col, stat in summary.columns]   # 2 段の列名を 1 段にする
print("\n種ごとの平均と標準偏差（転置して表示）:")
print(summary.round(3).T)

print("\n花弁の長さと幅の min / median / max:")
iris.groupby("species")[["petal_length", "petal_width"]].agg(["min", "median", "max"])

**結果の読み方**

- 3 種とも 50 個ずつあり、バランスの取れたデータです。
- `agg(["mean", "std"])` を DataFrame 全体に適用すると、列名が 2 段（変数名 × 統計量）の MultiIndex になります。扱いにくいときは `f"{col}_{stat}"` のように結合して 1 段にすると便利です。
- petal_length は setosa（1.0〜1.9 cm）と virginica（4.5〜6.9 cm）で範囲が全く重なりません。この変数だけで setosa を完全に見分けられることが分かります。
- このような「グループごとの要約表」は、レポートの最初に載せる基本の表（Table 1）です。

## 4. 確率分布

統計的推測（推定・検定）は「データがある確率分布から生まれた」という考え方に基づきます。
`scipy.stats` には主要な分布がそろっていて、どの分布でも同じメソッドが使えます。

| メソッド | 意味 |
|---|---|
| `pmf(k)` / `pdf(x)` | 確率質量関数（離散）/ 確率密度関数（連続） |
| `cdf(x)` | 累積分布関数 P(X ≤ x) |
| `sf(x)` | 生存関数 1 − cdf(x) = P(X > x) |
| `ppf(q)` | cdf の逆関数（分位点）: 下から q の位置の値 |
| `mean()`, `var()`, `std()` | 期待値、分散、標準偏差 |
| `rvs(size=, random_state=)` | 乱数を生成 |

### 例題 18: 二項分布（コイン投げ）

**背景**: 公正なコインを 10 回投げます。表がちょうど 5 回出る確率、8 回以上出る確率はいくつでしょうか。

「成功確率 p の試行を n 回繰り返したときの成功回数」は **二項分布 B(n, p)** に従います。

In [ ]:
n, p = 10, 0.5
k = np.arange(0, n + 1)
pmf = stats.binom.pmf(k, n, p)

print(f"表がちょうど 5 回出る確率 P(X=5)  = {stats.binom.pmf(5, n, p):.4f}")
print(f"表が 8 回以上出る確率   P(X>=8) = {stats.binom.sf(7, n, p):.4f}   # sf(7) = 1 - cdf(7)")
print(f"表が 3 回以下の確率     P(X<=3) = {stats.binom.cdf(3, n, p):.4f}")
print(f"期待値 np = {stats.binom.mean(n, p):.1f}, 分散 np(1-p) = {stats.binom.var(n, p):.2f}")

# 表が出にくいコイン (p = 0.3) と比べる
pmf_biased = stats.binom.pmf(k, n, 0.3)

plt.figure(figsize=(6, 3.8))
plt.bar(k - 0.2, pmf, width=0.4, label="p = 0.5", color="steelblue")
plt.bar(k + 0.2, pmf_biased, width=0.4, label="p = 0.3", color="darkorange")
plt.xticks(k)
plt.xlabel("表が出る回数 k")
plt.ylabel("確率 P(X = k)")
plt.title("二項分布 B(10, p) の確率質量関数")
plt.legend()
plt.show()

**結果の読み方**

- P(X=5) ≈ 0.246。「10 回投げてちょうど半分が表」になる確率は 4 回に 1 回程度しかありません。
- P(X≥8) ≈ 0.055。公正なコインでも 8 回以上表が出ることは約 5.5% の確率で起こります。この「珍しさの度合い」が仮説検定の p 値の考え方につながります。
- `sf(7)` は「7 より大きい」= 8 以上を意味します。離散分布では `cdf` / `sf` の境界の扱いに注意してください（`P(X>=8) = sf(7)`、`P(X<=3) = cdf(3)`）。
- p を 0.3 にすると山が左に寄り、左右非対称になります。

### 例題 19: ポアソン分布（1 時間あたりの来客数）

**背景**: ある店には 1 時間に平均 4 人の客が来ます。1 時間に誰も来ない確率、8 人以上来る確率はいくつでしょうか。

「一定時間（または面積）内に起こるまれな事象の回数」は **ポアソン分布 Po(λ)** に従います（λ = 平均回数）。
理論値と、乱数による 1 万時間分のシミュレーションを比べてみます。

In [ ]:
lam = 4   # 1 時間あたり平均 4 人
k = np.arange(0, 15)
pmf = stats.poisson.pmf(k, lam)

print(f"誰も来ない確率   P(X=0)  = {stats.poisson.pmf(0, lam):.4f}")
print(f"ちょうど 4 人     P(X=4)  = {stats.poisson.pmf(4, lam):.4f}")
print(f"8 人以上         P(X>=8) = {stats.poisson.sf(7, lam):.4f}")
print(f"期待値 = 分散 = λ = {stats.poisson.mean(lam):.0f}")

rng = np.random.default_rng(18)
sim = rng.poisson(lam, size=10000)
sim_freq = pd.Series(sim).value_counts(normalize=True).sort_index()
print(f"\nシミュレーション: 平均 {sim.mean():.3f}, 分散 {sim.var():.3f}, P(X>=8) ≈ {(sim >= 8).mean():.4f}")

plt.figure(figsize=(6, 3.8))
plt.bar(k - 0.2, pmf, width=0.4, label="理論値 (pmf)", color="steelblue")
plt.bar(sim_freq.index + 0.2, sim_freq.values, width=0.4, label="シミュレーション (1 万時間)", color="darkorange")
plt.xlabel("1 時間あたりの来客数 k")
plt.ylabel("確率")
plt.title(f"ポアソン分布 Po({lam})")
plt.legend()
plt.show()

**結果の読み方**

- 誰も来ない確率は約 1.8%、8 人以上来る確率は約 5.1% です。「平均 4 人」でも 8 人以上来る時間帯がおよそ 20 時間に 1 回あるので、その分の人員を考える、といった使い方ができます。
- ポアソン分布は **平均と分散が等しい**（どちらも λ）のが特徴です。実データで分散が平均よりずっと大きい場合（過分散）はポアソン分布が合っていません。
- シミュレーションの相対度数は理論値とよく一致しています。分布の性質を確かめたいときは、このように乱数で試すのが手軽で確実です。

### 例題 20: 正規分布（確率の計算と偏差値）

**背景**: あるテストの点数は平均 60 点、標準偏差 10 点の正規分布に従うとします。

1. 50 点以上 70 点以下の人は全体の何 % か
2. 上位 10% に入るには何点必要か
3. 75 点の偏差値と、それが上位何 % か

`stats.norm(loc=平均, scale=標準偏差)` で正規分布オブジェクトを作り、`cdf` / `sf` / `ppf` で計算します。

In [ ]:
mu, sigma = 60, 10
dist = stats.norm(loc=mu, scale=sigma)

print(f"50〜70 点 (平均 ± 1σ) の割合 : {dist.cdf(70) - dist.cdf(50):.4f}")
print(f"40〜80 点 (平均 ± 2σ) の割合 : {dist.cdf(80) - dist.cdf(40):.4f}")
print(f"30〜90 点 (平均 ± 3σ) の割合 : {dist.cdf(90) - dist.cdf(30):.4f}")
print(f"80 点以上の割合              : {dist.sf(80):.4f}")
print(f"上位 10% に入るための点数     : {dist.ppf(0.90):.1f} 点")
z = (75 - mu) / sigma
print(f"75 点 → z = {z:.2f}, 偏差値 = {50 + 10 * z:.0f}, 上位 {dist.sf(75) * 100:.1f}%")
print(f"偏差値 70 以上 (z >= 2) の割合  : {stats.norm.sf(2) * 100:.2f}%")

x = np.linspace(20, 100, 400)
plt.figure(figsize=(6, 3.8))
plt.plot(x, dist.pdf(x), color="navy")
mask = (x >= 50) & (x <= 70)
plt.fill_between(x[mask], dist.pdf(x[mask]), color="skyblue", alpha=0.6, label="50〜70 点 (68.3%)")
plt.axvline(dist.ppf(0.90), color="red", linestyle="--", label=f"上位 10% の境界 {dist.ppf(0.90):.1f} 点")
plt.xlabel("点数")
plt.ylabel("確率密度")
plt.title("正規分布 N(60, 10²)")
plt.legend(fontsize=8)
plt.show()

**結果の読み方**

- 平均 ± 1σ に約 68.3%、± 2σ に約 95.4%、± 3σ に約 99.7% のデータが入ります（**68-95-99.7 ルール**）。正規分布ではこの数字を覚えておくと便利です。
- 上位 10% の境界は約 72.8 点。`ppf(0.90)` は「下から 90% の位置」を返します。
- 75 点は偏差値 65 で上位約 6.7%。偏差値 70（z = 2）以上は上位約 2.3% です。
- ただし、これらは **点数が正規分布に従う場合** の話です。実際のテストの点数は天井効果（満点付近に集まる）などで歪んでいることも多く、その場合は偏差値と実際の順位はずれます。

### 例題 21: 一様分布と指数分布

**背景**:

1. 10 分間隔で来るバスの停留所に、時刻を気にせず到着します。待ち時間は 0〜10 分の **一様分布** に従います。3 分以内にバスが来る確率は？
2. コールセンターには平均 5 分に 1 回電話がかかってきます。「次の電話までの時間」は平均 5 分の **指数分布** に従います。10 分以上電話が来ない確率は？

指数分布はポアソン分布と表裏一体で、「事象が起こる間隔」を表します。`scale` に平均（= 1/λ）を渡します。

In [ ]:
rng = np.random.default_rng(20)

wait = stats.uniform(loc=0, scale=10)    # 0〜10 分の一様分布
print(f"[一様分布] 3 分以内にバスが来る確率: {wait.cdf(3):.3f}, 平均待ち時間: {wait.mean():.1f} 分, 標準偏差: {wait.std():.2f} 分")

call = stats.expon(scale=5)              # 平均 5 分の指数分布
print(f"[指数分布] 次の電話まで 10 分以上: {call.sf(10):.3f}, 2 分以内: {call.cdf(2):.3f}, 中央値: {call.median():.2f} 分")

sim_wait = rng.uniform(0, 10, size=10000)
sim_call = rng.exponential(scale=5, size=10000)
print(f"\nシミュレーション: 一様分布の平均 {sim_wait.mean():.2f}, 指数分布の平均 {sim_call.mean():.2f}, "
      f"10 分以上の割合 {(sim_call > 10).mean():.3f}")

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
x1 = np.linspace(-1, 11, 300)
axes[0].hist(sim_wait, bins=30, density=True, color="lightgray", label="シミュレーション")
axes[0].plot(x1, wait.pdf(x1), color="navy", label="理論 pdf")
axes[0].set_title("一様分布 U(0, 10)")
axes[0].set_xlabel("待ち時間 (分)")
x2 = np.linspace(0, 30, 300)
axes[1].hist(sim_call, bins=40, density=True, color="lightgray", label="シミュレーション")
axes[1].plot(x2, call.pdf(x2), color="darkred", label="理論 pdf")
axes[1].set_title("指数分布（平均 5 分）")
axes[1].set_xlabel("次の電話までの時間 (分)")
for ax in axes:
    ax.set_ylabel("確率密度")
    ax.legend()
plt.tight_layout()
plt.show()

**結果の読み方**

- 一様分布では確率が区間の長さに比例するので、3 分以内の確率は 3/10 = 0.3 です。
- 指数分布では 10 分以上待つ確率は e^(−10/5) ≈ 0.135。平均は 5 分ですが中央値は約 3.5 分で、**平均 > 中央値** の右に歪んだ分布です。
- 指数分布には「これまでどれだけ待ったかは、これから待つ時間に影響しない」という **無記憶性** があります。「もう 10 分も来ていないからそろそろ来るはず」は指数分布の世界では成り立ちません。
- 連続分布では `pdf` の値そのものは確率ではなく、区間で積分（`cdf` の差）して初めて確率になります。

### 例題 22: 大数の法則（乱数シミュレーション）

**背景**: サイコロを振り続けて出た目の平均を記録します。試行回数が増えると、平均は理論値 3.5 に近づくでしょうか。

これが **大数の法則** です。「平均は試行を増やせば真の値に収束する」という、統計的推測の土台になる性質です。

In [ ]:
rng = np.random.default_rng(21)
n_max = 5000
rolls = rng.integers(1, 7, size=n_max)               # 1〜6 の目
running_mean = np.cumsum(rolls) / np.arange(1, n_max + 1)

print("出た目の分布:", np.bincount(rolls)[1:])
for n in [10, 100, 1000, 5000]:
    print(f"n = {n:5d} 回までの平均: {running_mean[n - 1]:.3f}  (理論値 3.5 との差 {running_mean[n - 1] - 3.5:+.3f})")

plt.figure(figsize=(6.5, 3.8))
plt.plot(np.arange(1, n_max + 1), running_mean, lw=1)
plt.axhline(3.5, color="red", linestyle="--", label="理論値 3.5")
plt.xscale("log")
plt.xlabel("試行回数 n（対数目盛）")
plt.ylabel("それまでの平均")
plt.title("大数の法則: サイコロの目の平均は 3.5 に近づく")
plt.legend()
plt.show()

**結果の読み方**

- 最初の 10 回では平均が 3.5 から大きく外れることがありますが、回数を重ねるほど 3.5 に近づいていきます。
- 収束のスピードは √n に反比例します（次の例題の標準誤差）。精度を 10 倍にするには試行回数を 100 倍にする必要があります。
- 大数の法則は「長期的な平均」についての法則で、「これまで小さい目が続いたから次は大きい目が出やすい」という意味ではありません（**ギャンブラーの誤謬**）。

## 5. 標本と推定

知りたいのは母集団（全体）の性質ですが、手に入るのは標本（一部）だけです。
標本から母集団の値（母平均、母比率など）を **推定** し、その **不確かさ** を信頼区間で表します。

### 例題 23: 中心極限定理（サイコロの平均の分布）

**背景**: サイコロを n 個振って出た目の平均を計算する、という実験を 5000 回繰り返します。n = 1, 2, 10, 30 のとき「平均」のヒストグラムはどんな形になるでしょうか。

サイコロ 1 個の目は一様分布（正規分布ではない）ですが、**標本平均の分布は n が大きくなると正規分布に近づく** ──これが **中心極限定理** です。

In [ ]:
rng = np.random.default_rng(22)
sizes = [1, 2, 10, 30]
n_trials = 5000

fig, axes = plt.subplots(1, 4, figsize=(11, 2.8))
for ax, n in zip(axes, sizes):
    means = rng.integers(1, 7, size=(n_trials, n)).mean(axis=1)
    ax.hist(means, bins=30, color="steelblue", edgecolor="white")
    ax.set_title(f"n = {n}\n平均 {means.mean():.2f}, SD {means.std():.3f}")
    ax.set_xlabel("標本平均")
axes[0].set_ylabel("度数")
plt.tight_layout()
plt.show()

sigma = np.sqrt(35 / 12)   # サイコロ 1 個の目の標準偏差
print(f"サイコロ 1 個の標準偏差 σ = {sigma:.3f}")
for n in sizes:
    print(f"  n = {n:2d}: 標本平均の標準偏差の理論値 σ/√n = {sigma / np.sqrt(n):.3f}")

**結果の読み方**

- n = 1 では 1〜6 が同じ高さの平らな分布ですが、n = 2 で三角形、n = 10 以上ではきれいな釣鐘型（正規分布）になります。
- ヒストグラムの中心は常に 3.5（母平均）で、幅は n が大きいほど狭くなります。グラフに表示した SD が理論値 σ/√n とほぼ一致していることを確認してください。
- 元のデータが正規分布でなくても、**標本平均は正規分布に近づく**。だから t 検定や信頼区間のような正規分布に基づく手法が広く使えるのです。
- 目安として n ≥ 30 なら多くの場合で正規近似が使えますが、元の分布が極端に歪んでいる場合はもっと必要です。

### 例題 24: 標準誤差と母平均の信頼区間（t 分布）

**背景**: ある地域の成人男性 25 人の身長を測ったところ、平均 170 cm 前後でした。この地域全体（母集団）の平均身長はどの範囲にありそうでしょうか。

- **標準誤差 (SE)** = s / √n: 標本平均の「ばらつき」。母平均の推定の精度を表す。
- **95% 信頼区間** = 標本平均 ± t(0.975, n−1) × SE。母標準偏差が未知なので正規分布ではなく **t 分布** を使う。

後半では「95% 信頼区間」の意味をシミュレーションで確かめます。母平均 170 cm と分かっている母集団から 25 人の標本を取って信頼区間を作る作業を 50 回繰り返し、そのうち何回が母平均を含むかを数えます。

In [ ]:
rng = np.random.default_rng(23)
true_mu, true_sigma = 170, 6      # 本当の母平均・母標準偏差（普段は分からない）
sample = rng.normal(true_mu, true_sigma, size=25)

n = len(sample)
mean = sample.mean()
sd = sample.std(ddof=1)
se = sd / np.sqrt(n)              # 標準誤差
ci95 = stats.t.interval(0.95, df=n - 1, loc=mean, scale=se)
ci99 = stats.t.interval(0.99, df=n - 1, loc=mean, scale=se)
print(f"標本平均 {mean:.2f} cm, 標本標準偏差 {sd:.2f} cm, 標準誤差 {se:.3f} cm")
print(f"母平均の 95% 信頼区間: [{ci95[0]:.2f}, {ci95[1]:.2f}]  (幅 {ci95[1] - ci95[0]:.2f})")
print(f"母平均の 99% 信頼区間: [{ci99[0]:.2f}, {ci99[1]:.2f}]  (幅 {ci99[1] - ci99[0]:.2f})")
print(f"t 分布の 97.5% 点 (df={n - 1}) = {stats.t.ppf(0.975, df=n - 1):.3f}  (正規分布なら 1.960)")

# 「95% 信頼区間」の意味を確かめる
n_rep = 50
hits = 0
plt.figure(figsize=(6.5, 4))
for i in range(n_rep):
    s = rng.normal(true_mu, true_sigma, size=n)
    lo, hi = stats.t.interval(0.95, df=n - 1, loc=s.mean(), scale=s.std(ddof=1) / np.sqrt(n))
    contains = lo <= true_mu <= hi
    hits += contains
    plt.plot([lo, hi], [i, i], color="steelblue" if contains else "red", lw=1.5)
plt.axvline(true_mu, color="black", linestyle="--", label=f"母平均 {true_mu} cm")
plt.xlabel("身長 (cm)")
plt.ylabel("繰り返し番号")
plt.title(f"95% 信頼区間を {n_rep} 回作ると {hits} 回が母平均を含んだ")
plt.legend()
plt.show()

**結果の読み方**

- 95% 信頼区間は「母平均がこの範囲にある確率が 95%」ではなく、「**同じ手順で区間を作る作業を繰り返すと、95% の区間が母平均を含む**」という意味です。グラフでは 50 本のうち 47 本が母平均を含み、赤い区間（母平均を外した区間）は 3 本です。
- 信頼係数を 99% に上げると区間は広くなります。「確実さ」と「精度」はトレードオフです。
- 区間の幅は SE に比例し、SE は 1/√n で小さくなります。幅を半分にするには n を 4 倍にする必要があります。
- n が小さいときは t 分布の分位点（2.064）が正規分布（1.960）より大きくなり、区間が少し広がります。n が大きくなると両者はほぼ一致します。

### 例題 25: 母比率の信頼区間

**背景**: 有権者 400 人を無作為に選んで調査したところ、180 人（45%）がある政策に賛成でした。有権者全体の賛成率はどの範囲にありそうでしょうか。

比率の信頼区間にはいくつかの方法があります。

| 方法 | 特徴 |
|---|---|
| Wald 法（正規近似） | p̂ ± z × √(p̂(1−p̂)/n)。簡単だが n が小さいときや p̂ が 0, 1 に近いとき不正確 |
| Wilson 法 | 正規近似だが精度が高い。**実務ではこれを推奨** |
| Clopper-Pearson（正確法） | 二項分布に基づく。保守的（広め） |

In [ ]:
from statsmodels.stats.proportion import proportion_confint

count, nobs = 180, 400
p_hat = count / nobs
se = np.sqrt(p_hat * (1 - p_hat) / nobs)
z = stats.norm.ppf(0.975)
print(f"標本比率 p̂ = {p_hat:.3f}, 標準誤差 = {se:.4f}")
print(f"Wald 法          の 95% CI: [{p_hat - z * se:.3f}, {p_hat + z * se:.3f}]")

lo, hi = proportion_confint(count, nobs, alpha=0.05, method="wilson")
print(f"Wilson 法        の 95% CI: [{lo:.3f}, {hi:.3f}]")
ci = stats.binomtest(count, nobs).proportion_ci(confidence_level=0.95)
print(f"Clopper-Pearson  の 95% CI: [{ci.low:.3f}, {ci.high:.3f}]")

print("\nサンプルサイズと区間の幅 (p̂ = 0.45, Wilson 法):")
for n in [100, 400, 1600, 6400]:
    lo, hi = proportion_confint(int(0.45 * n), n, method="wilson")
    print(f"  n = {n:5d}: [{lo:.3f}, {hi:.3f}]  幅 {hi - lo:.3f}")

**結果の読み方**

- 賛成率の 95% 信頼区間はおよそ 40%〜50% です。「45% だから半数未満」と言い切るには区間が 50% をまたいでいるため根拠が足りない、と読めます。
- 3 つの方法の結果はこの例（n = 400、p̂ = 0.45）ではほぼ同じです。違いが出るのは n が小さいときや比率が 0 や 1 に近いときで、そのときは Wilson 法か正確法を使ってください。
- 世論調査で「誤差 ±3%」と言われるのは、n ≈ 1000 前後で p ≈ 0.5 のときの 95% 信頼区間の半幅が約 3% だからです。

### 例題 26: ブートストラップ信頼区間

**背景**: 1 回の来店あたりの購入金額（千円）を 40 人分記録しました。分布は右に大きく歪んでいるので、平均ではなく **中央値** の信頼区間を求めたい。しかし中央値の信頼区間には簡単な公式がありません。

**ブートストラップ法** は、手元の標本から **復元抽出で同じサイズの再標本を何千回も作り**、統計量の分布を直接シミュレーションする方法です。公式が分からない統計量にも使えます。

In [ ]:
rng = np.random.default_rng(25)
purchase = rng.lognormal(mean=1.5, sigma=0.7, size=40).round(1)
observed_median = np.median(purchase)
print(f"データ (千円): 平均 {purchase.mean():.2f}, 中央値 {observed_median:.2f}, 歪度 {stats.skew(purchase):.2f}")

n_boot = 5000
boot_medians = np.empty(n_boot)
for i in range(n_boot):
    resample = rng.choice(purchase, size=len(purchase), replace=True)   # 復元抽出
    boot_medians[i] = np.median(resample)

ci_low, ci_high = np.percentile(boot_medians, [2.5, 97.5])
print(f"ブートストラップ標準誤差: {boot_medians.std(ddof=1):.3f}")
print(f"中央値の 95% 信頼区間（パーセンタイル法）: [{ci_low:.2f}, {ci_high:.2f}]")

plt.figure(figsize=(6, 3.8))
plt.hist(boot_medians, bins=40, color="steelblue", edgecolor="white")
plt.axvline(observed_median, color="red", label=f"標本の中央値 {observed_median:.2f}")
plt.axvline(ci_low, color="gray", linestyle="--", label="95% 信頼区間")
plt.axvline(ci_high, color="gray", linestyle="--")
plt.xlabel("再標本の中央値 (千円)")
plt.ylabel("度数")
plt.title("ブートストラップによる中央値の分布")
plt.legend(fontsize=8)
plt.show()

**結果の読み方**

- 5000 個の再標本の中央値の分布から、2.5% 点と 97.5% 点を取ったものが 95% 信頼区間です（パーセンタイル法）。
- ブートストラップの分布は左右対称とは限りません。中央値のような統計量では、とびとびの値（元データの値のどれか）しか取らないためヒストグラムが櫛状になります。
- 手元の標本が母集団をよく代表していることが前提です。n が極端に小さい（10 未満など）と信頼できません。
- `scipy.stats.bootstrap` 関数を使うと、より精度の高い BCa 法などが利用できます。

## 6. 仮説検定

### 検定の考え方

仮説検定は「データが、ある仮説のもとでどれくらい珍しいか」を測る手続きです。

1. **帰無仮説 H0**（差がない、効果がない）と **対立仮説 H1**（差がある）を立てる。
2. H0 が正しいと仮定して、観測されたデータ以上に極端な結果が得られる確率 = **p 値** を計算する。
3. p 値が **有意水準 α**（慣習的に 0.05）より小さければ H0 を棄却し、「有意な差がある」と判断する。

| | H0 が真（本当は差がない） | H0 が偽（本当は差がある） |
|---|---|---|
| H0 を棄却する（差があると判断） | **第 1 種の誤り**（確率 α） | 正しい判断（確率 = 検出力 1−β） |
| H0 を棄却しない | 正しい判断 | **第 2 種の誤り**（確率 β） |

p 値についてよくある誤解:

- p 値は「H0 が正しい確率」**ではありません**。
- p ≥ 0.05 は「差がないことの証明」**ではありません**。「差があるとは言えない」だけです。
- p < 0.05 でも差が実用上意味のある大きさとは限りません（効果量を見る: 例題 36）。
- 検定の種類はデータの種類と問いによって決まります（まとめの表を参照）。

### 例題 27: 1 標本 t 検定

**背景**: 「内容量 500 g」と表示された商品を 12 個計量しました。平均は 500 g と言えるでしょうか。

**データ**: `498.2, 501.5, 497.8, 499.1, 502.3, 496.5, 498.9, 500.4, 497.2, 499.6, 498.0, 497.5`（g）

- H0: 母平均 μ = 500、H1: μ ≠ 500（両側検定）
- `stats.ttest_1samp(データ, popmean=500)`

In [ ]:
weights = [498.2, 501.5, 497.8, 499.1, 502.3, 496.5, 498.9, 500.4, 497.2, 499.6, 498.0, 497.5]
n = len(weights)
res = stats.ttest_1samp(weights, popmean=500)
ci = res.confidence_interval(0.95)

print(f"標本平均 {np.mean(weights):.2f} g, 標本標準偏差 {np.std(weights, ddof=1):.2f} g, n = {n}")
print(f"t 統計量 = {res.statistic:.3f}, 自由度 = {n - 1}, p 値 = {res.pvalue:.4f}")
print(f"母平均の 95% 信頼区間: [{ci.low:.2f}, {ci.high:.2f}]")
if res.pvalue < 0.05:
    print("→ 有意水準 5% で H0 (μ = 500) を棄却: 平均は 500 g と異なると言える")
else:
    print("→ H0 は棄却されない: 平均が 500 g と異なるとは言えない")

# 片側検定: 最初から「500 g より少ないのでは」と疑っていた場合
res_less = stats.ttest_1samp(weights, popmean=500, alternative="less")
print(f"\n片側検定 (H1: μ < 500) の p 値 = {res_less.pvalue:.4f}")

**結果の読み方**

- 標本平均は 498.9 g で 500 g より少し軽いですが、両側検定の p 値は約 0.057 で、有意水準 5% では **棄却できません**。95% 信頼区間 [497.8, 500.0] が 500 を（ぎりぎり）含んでいることとも整合しています。
- 片側検定の p 値は約 0.028 で、こちらなら棄却できます。しかし **データを見てから片側に変える** のは不正です。片側検定を使うのは、分析前から方向が決まっている場合に限ります。
- p = 0.057 と p = 0.049 に本質的な違いはありません。0.05 という境界は慣習にすぎず、「わずかに有意でない」結果はサンプルを増やして再検証するのが誠実な態度です。
- t 検定は「データが正規分布に近い」ことを仮定します。n が小さいときは特に、例題 31 の方法で確認しましょう。

### 例題 28: 2 標本 t 検定（Welch の t 検定）

**背景**: 従来の教え方 (A) と新しい教え方 (B) でそれぞれ別の生徒に授業をし、テストの点数を比べます。B の方が高いと言えるでしょうか。

**データ**:
- A (15 人): `68, 72, 75, 61, 80, 70, 66, 74, 69, 77, 63, 71, 73, 65, 70`
- B (14 人): `78, 74, 85, 70, 88, 79, 76, 82, 90, 73, 81, 77, 84, 72`

独立した 2 群の平均を比べるには `stats.ttest_ind` を使います。`equal_var=False` にすると 2 群の分散が等しいと仮定しない **Welch の t 検定** になります。分散が等しいかどうか事前に検定する方法もありますが、**最初から Welch を使う** のが現在の標準的な推奨です。

In [ ]:
group_a = [68, 72, 75, 61, 80, 70, 66, 74, 69, 77, 63, 71, 73, 65, 70]
group_b = [78, 74, 85, 70, 88, 79, 76, 82, 90, 73, 81, 77, 84, 72]

res = stats.ttest_ind(group_a, group_b, equal_var=False)   # Welch の t 検定
ci = res.confidence_interval(0.95)
print(f"A: 平均 {np.mean(group_a):.2f} (SD {np.std(group_a, ddof=1):.2f}, n = {len(group_a)})")
print(f"B: 平均 {np.mean(group_b):.2f} (SD {np.std(group_b, ddof=1):.2f}, n = {len(group_b)})")
print(f"平均の差 (A - B) = {np.mean(group_a) - np.mean(group_b):.2f},  95% CI: [{ci.low:.2f}, {ci.high:.2f}]")
print(f"t = {res.statistic:.3f}, 自由度 ≈ {res.df:.1f}, p 値 = {res.pvalue:.5f}")

# 参考: 等分散を仮定した Student の t 検定
res_student = stats.ttest_ind(group_a, group_b, equal_var=True)
print(f"（参考）Student の t 検定: t = {res_student.statistic:.3f}, p 値 = {res_student.pvalue:.5f}")

plt.figure(figsize=(5, 3.5))
plt.boxplot([group_a, group_b])
plt.xticks([1, 2], ["A: 従来", "B: 新方式"])
plt.ylabel("点数")
plt.title("2 群のテスト点数")
plt.show()

**結果の読み方**

- B の平均は A より約 9 点高く、p 値は 0.001 未満です。有意水準 5%（1% でも）で「2 群の平均に差がある」と言えます。
- 平均の差の 95% 信頼区間 [約 −13, 約 −5] は 0 を含みません。**信頼区間が 0 を含まないことと p < 0.05 は同じこと** です。信頼区間は「差の大きさ」まで教えてくれるので、p 値と一緒に必ず報告しましょう。
- Welch の t 検定では自由度が整数でない値になります。Student の t 検定と結果がほとんど同じなのは、この例では 2 群の分散が近いからです。
- 「差がある」ことと「新方式のおかげ」は別の話です。生徒を無作為に 2 群に分けていなければ、もともと成績の良い生徒が B に集まっていた可能性を否定できません。

### 例題 29: 対応のある t 検定

**背景**: 10 人の社員に研修を行い、研修前と研修後に同じテストを受けてもらいました。研修で点数は上がったでしょうか。

**データ**:
- 研修前: `62, 70, 55, 68, 74, 60, 65, 72, 58, 66`
- 研修後: `68, 74, 60, 70, 79, 63, 70, 75, 64, 70`

同じ人を 2 回測ったデータは **対応のあるデータ** です。各人の「後 − 前」の差を取り、その平均が 0 かどうかを検定します（`stats.ttest_rel`）。
これを独立な 2 群として扱ってしまうと、個人差という大きなばらつきが混ざって検出力が落ちます。

In [ ]:
before = np.array([62, 70, 55, 68, 74, 60, 65, 72, 58, 66])
after = np.array([68, 74, 60, 70, 79, 63, 70, 75, 64, 70])
diff = after - before

res = stats.ttest_rel(after, before)
print(f"研修前の平均 {before.mean():.1f}, 研修後の平均 {after.mean():.1f}")
print(f"差 (後 - 前): {diff.tolist()}")
print(f"差の平均 {diff.mean():.2f}, 差の SD {diff.std(ddof=1):.2f}")
print(f"対応のある t 検定: t = {res.statistic:.3f}, 自由度 = {len(diff) - 1}, p 値 = {res.pvalue:.6f}")

# 誤って「独立な 2 群」として検定すると…
res_ind = stats.ttest_ind(after, before, equal_var=False)
print(f"（誤用）対応を無視した Welch の t 検定: t = {res_ind.statistic:.3f}, p 値 = {res_ind.pvalue:.4f}")

plt.figure(figsize=(5, 3.8))
for b, a in zip(before, after):
    plt.plot([0, 1], [b, a], marker="o", color="steelblue", alpha=0.7)
plt.xticks([0, 1], ["研修前", "研修後"])
plt.xlim(-0.3, 1.3)
plt.ylabel("点数")
plt.title("同じ人の研修前後の点数")
plt.show()

**結果の読み方**

- 全員の点数が上がっており（差はすべて正）、差の平均は 4.3 点、p 値は 0.0001 未満で明確に有意です。
- 同じデータを独立 2 群として検定すると p ≈ 0.14 で有意になりません。個人差（55 点の人と 74 点の人）が「ノイズ」として扱われてしまうからです。**対応があるなら必ず対応のある検定を使う** ことが重要です。
- 対応のある t 検定は「差の 1 標本 t 検定」と同じです。`stats.ttest_1samp(diff, 0)` でも同じ結果になります。
- 研修前後の比較には「時間が経てば自然に上がる（練習効果）」という交絡の可能性があります。因果を主張するには、研修を受けない対照群との比較が必要です。

### 例題 30: 母比率の検定（1 標本・2 標本）

**背景**:

1. 新しいランディングページに 1200 人が訪れ、78 人が申し込みました（成約率 6.5%）。目標の 5% を上回っていると言えるでしょうか。
2. A/B テスト: ボタンの色が違う 2 つのページに 1000 人ずつを割り当てたところ、申し込みは A が 60 人、B が 85 人でした。差があるでしょうか。

比率の検定には statsmodels の `proportions_ztest` を使います（正規近似）。1 標本なら二項分布に基づく正確な検定 `stats.binomtest` も使えます。

In [ ]:
from statsmodels.stats.proportion import proportions_ztest

# (1) 1 標本: 成約率は 5% より高いか（片側）
conversions, visitors = 78, 1200
stat, p = proportions_ztest(count=conversions, nobs=visitors, value=0.05, alternative="larger", prop_var=0.05)
print(f"成約率 = {conversions / visitors:.4f}")
print(f"z 検定: z = {stat:.3f}, p 値 = {p:.4f}  (H1: p > 0.05)")
res_binom = stats.binomtest(conversions, visitors, p=0.05, alternative="greater")
print(f"二項検定（正確な検定）: p 値 = {res_binom.pvalue:.4f}")

# (2) 2 標本: A/B テスト
count = np.array([60, 85])
nobs = np.array([1000, 1000])
stat2, p2 = proportions_ztest(count, nobs)
print(f"\nA/B テスト: A = {count[0] / nobs[0]:.1%}, B = {count[1] / nobs[1]:.1%}, 差 = {(count[1] - count[0]) / 1000:.1%} ポイント")
print(f"z = {stat2:.3f}, p 値 = {p2:.4f}  (両側)")

# 同じ問いをカイ二乗検定で解くこともできる（例題 34 参照）
table = np.array([[60, 940], [85, 915]])
chi2, p_chi2, _, _ = stats.chi2_contingency(table, correction=False)
print(f"（参考）カイ二乗検定: χ² = {chi2:.3f}, p 値 = {p_chi2:.4f}  ← z² = {stat2 ** 2:.3f} と一致")

**結果の読み方**

- (1) z ≈ 2.38、p ≈ 0.009 で、成約率は 5% より有意に高いと言えます。`prop_var=0.05` は帰無仮説の比率を使って標準誤差を計算する指定で、教科書の式と一致します。正確な二項検定でも p ≈ 0.013 と近い値です。
- (2) B の成約率は A より 2.5 ポイント高く、p ≈ 0.03 で有意です。ただし差の信頼区間も見て、「B に切り替える価値がある差か」をビジネス上の観点で判断してください。
- 2 × 2 の分割表に対するカイ二乗検定（連続性補正なし）は、2 標本の比率の z 検定と数学的に同じものです（χ² = z²）。
- A/B テストでは「途中で何度も p 値を見て、有意になった時点でやめる」と第 1 種の誤りが激増します。サンプルサイズを事前に決めておきましょう（例題 38）。

### 例題 31: 正規性の検定（Shapiro-Wilk）と Q-Q プロット

**背景**: t 検定や分散分析はデータが正規分布に近いことを仮定しています。手元のデータが正規分布から大きく外れていないかを調べたい。

- **Shapiro-Wilk 検定**: H0 = 「データは正規分布に従う」。p < 0.05 なら正規分布とは言えない。
- **Q-Q プロット**: データの分位点と正規分布の理論分位点を散布図にしたもの。正規分布なら点が直線上に並ぶ。

正規分布からの標本と、指数分布（右に歪んだ分布）からの標本で比べます。

In [ ]:
rng = np.random.default_rng(30)
normal_sample = rng.normal(50, 10, size=60)
skewed_sample = rng.exponential(scale=10, size=60)

fig, axes = plt.subplots(1, 2, figsize=(9, 3.8))
for ax, (name, x) in zip(axes, [("正規分布からの標本", normal_sample), ("指数分布からの標本", skewed_sample)]):
    w, p = stats.shapiro(x)
    print(f"{name}: Shapiro-Wilk W = {w:.4f}, p 値 = {p:.4g}, 歪度 = {stats.skew(x):.2f}")
    stats.probplot(x, dist="norm", plot=ax)
    ax.set_title(f"{name}\nShapiro-Wilk p = {p:.3g}")
    ax.set_xlabel("理論分位点（正規分布）")
    ax.set_ylabel("観測値の分位点")
plt.tight_layout()
plt.show()

**結果の読み方**

- 正規分布からの標本は p 値が大きく（0.05 以上）、Q-Q プロットの点はほぼ直線上に乗ります。
- 指数分布からの標本は p 値が非常に小さく、Q-Q プロットは右端で上に反り上がる（右の裾が長い）形になります。
- 正規性の検定には注意点があります。**n が大きいと些細なずれでも有意になり、n が小さいと大きなずれでも有意になりません**。検定の結果だけでなく、Q-Q プロットやヒストグラムで「実用上問題になるずれか」を判断してください。
- 正規性が疑わしいときの選択肢: (1) 対数変換などで正規分布に近づける、(2) ノンパラメトリック検定を使う（次の例題）、(3) n が十分大きければ中心極限定理により t 検定をそのまま使う。

### 例題 32: Mann-Whitney の U 検定（ノンパラメトリック検定）

**背景**: 2 種類の Web ページデザイン A, B について、訪問者 30 人ずつのページ滞在時間（秒）を測りました。滞在時間は右に大きく歪んだ分布です。B の方が長いと言えるでしょうか。

**Mann-Whitney の U 検定** は、値そのものではなく **順位** を使うので、分布の形を仮定せず、外れ値にも強い検定です。「2 群の分布の位置がずれているか」を調べます。

In [ ]:
rng = np.random.default_rng(231)
design_a = rng.lognormal(mean=3.0, sigma=0.8, size=30).round(1)
design_b = rng.lognormal(mean=3.6, sigma=0.8, size=30).round(1)
print(f"A: 中央値 {np.median(design_a):6.1f} 秒, 平均 {design_a.mean():6.1f} 秒, 最大 {design_a.max():6.1f} 秒")
print(f"B: 中央値 {np.median(design_b):6.1f} 秒, 平均 {design_b.mean():6.1f} 秒, 最大 {design_b.max():6.1f} 秒")

res = stats.mannwhitneyu(design_a, design_b, alternative="two-sided")
print(f"\nMann-Whitney の U 検定: U = {res.statistic:.1f}, p 値 = {res.pvalue:.4f}")
res_t = stats.ttest_ind(design_a, design_b, equal_var=False)
print(f"（参考）そのまま Welch の t 検定: p 値 = {res_t.pvalue:.4f}")
res_log = stats.ttest_ind(np.log(design_a), np.log(design_b), equal_var=False)
print(f"（参考）対数変換してから t 検定: p 値 = {res_log.pvalue:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
axes[0].hist(design_a, bins=15, alpha=0.6, label="A")
axes[0].hist(design_b, bins=15, alpha=0.6, label="B")
axes[0].set_xlabel("滞在時間 (秒)")
axes[0].set_ylabel("度数")
axes[0].set_title("元のデータ（右に歪んでいる）")
axes[1].hist(np.log(design_a), bins=12, alpha=0.6, label="A")
axes[1].hist(np.log(design_b), bins=12, alpha=0.6, label="B")
axes[1].set_xlabel("log(滞在時間)")
axes[1].set_title("対数変換後（正規分布に近い）")
for ax in axes:
    ax.legend()
plt.tight_layout()
plt.show()

**結果の読み方**

- 中央値は B の方が約 18 秒長く、U 検定の p 値は約 0.005 で有意です。
- 同じデータをそのまま t 検定にかけると p ≈ 0.09 で有意になりません。少数の極端に長い滞在時間が平均と分散を大きくし、差を検出しにくくしているためです。対数変換すると分布が正規分布に近づき、t 検定の p 値も U 検定とほぼ同じになります。
- U 検定の帰無仮説は厳密には「平均が等しい」ではなく「一方の群の値が他方より大きい確率が 1/2」です。分布の形が 2 群で同じなら「中央値の差の検定」と解釈できます。
- 対応のあるデータのノンパラメトリック版は **Wilcoxon の符号順位検定**（`stats.wilcoxon`）、3 群以上は **Kruskal-Wallis 検定**（`stats.kruskal`）です。

### 例題 33: カイ二乗検定（適合度検定）

**背景**: サイコロを 120 回振ったところ、各目の出た回数は `25, 17, 15, 23, 24, 16` でした。このサイコロは公正（各目 1/6）と言えるでしょうか。

**適合度検定** は「観測度数が、理論的に期待される度数の比率と合っているか」を調べます。

$$\chi^2 = \sum \frac{(観測度数 - 期待度数)^2}{期待度数}$$

続けて、血液型の分布が「A : O : B : AB = 4 : 3 : 2 : 1」に従うかも調べます。

In [ ]:
observed = np.array([25, 17, 15, 23, 24, 16])
expected = np.full(6, observed.sum() / 6)          # 公正なら各目 20 回
res = stats.chisquare(f_obs=observed, f_exp=expected)
print("観測度数:", observed)
print("期待度数:", expected)
print(f"χ² = {res.statistic:.3f}, 自由度 = {len(observed) - 1}, p 値 = {res.pvalue:.4f}")

# 期待比率が等しくない例: 血液型 A:O:B:AB = 4:3:2:1
blood_obs = np.array([88, 60, 38, 14])
blood_exp = blood_obs.sum() * np.array([0.4, 0.3, 0.2, 0.1])
res2 = stats.chisquare(f_obs=blood_obs, f_exp=blood_exp)
print(f"\n血液型 (n = {blood_obs.sum()}): 期待度数 {blood_exp}")
print(f"χ² = {res2.statistic:.3f}, 自由度 = 3, p 値 = {res2.pvalue:.4f}")

plt.figure(figsize=(5.5, 3.5))
x = np.arange(1, 7)
plt.bar(x - 0.2, observed, width=0.4, label="観測")
plt.bar(x + 0.2, expected, width=0.4, label="期待（公正なサイコロ）")
plt.xlabel("サイコロの目")
plt.ylabel("回数")
plt.title("適合度検定")
plt.legend()
plt.show()

**結果の読み方**

- サイコロ: χ² = 5.0、自由度 5、p ≈ 0.42。目の出方のばらつきは偶然の範囲内で、「公正でない」とは言えません。
- 血液型: χ² = 2.7、p ≈ 0.44。4 : 3 : 2 : 1 の比率と矛盾しません。
- 自由度は「カテゴリ数 − 1」です。
- カイ二乗検定は近似なので、**期待度数が 5 未満のセル** があると不正確です。その場合はカテゴリをまとめるか、正確検定を使います。
- 「棄却されない = 公正であることが証明された」ではありません。120 回程度では小さな偏りは検出できません（検出力の問題）。

### 例題 34: カイ二乗検定（独立性の検定）

**背景**: 200 人の顧客に 3 つの商品 X, Y, Z のどれが好きかを尋ね、性別ごとに集計しました。性別と商品の好みに関連はあるでしょうか。

| | 商品 X | 商品 Y | 商品 Z |
|---|---|---|---|
| 男性 | 45 | 30 | 25 |
| 女性 | 20 | 40 | 40 |

**独立性の検定** は、クロス集計表の 2 つの変数が独立（無関係）かどうかを調べます。`stats.chi2_contingency` は期待度数も返してくれます。
関連の強さは **Cramér の V**（0〜1）で測ります。

In [ ]:
table = pd.DataFrame([[45, 30, 25], [20, 40, 40]],
                     index=["男性", "女性"], columns=["商品X", "商品Y", "商品Z"])
print("観測度数:")
print(table)

chi2, p, dof, expected = stats.chi2_contingency(table)
print(f"\nχ² = {chi2:.3f}, 自由度 = {dof}, p 値 = {p:.5f}")
print("期待度数（性別と好みが独立なら期待される度数）:")
print(pd.DataFrame(expected, index=table.index, columns=table.columns).round(1))

n_total = table.to_numpy().sum()
cramers_v = np.sqrt(chi2 / (n_total * (min(table.shape) - 1)))
print(f"\nCramér の V = {cramers_v:.3f}  （目安: 0.1 弱い, 0.3 中程度, 0.5 強い）")

print("\n行方向の割合:")
print((table.div(table.sum(axis=1), axis=0)).round(3))

**結果の読み方**

- p 値は 0.001 未満で、性別と商品の好みには **統計的に有意な関連がある** と言えます。男性は商品 X、女性は商品 Y, Z を好む傾向です。
- 自由度は (行数 − 1) × (列数 − 1) = 1 × 2 = 2 です。
- 期待度数と観測度数を見比べると、どのセルが乖離しているか（どこに関連の原因があるか）が分かります。
- Cramér の V ≈ 0.27 は「弱〜中程度の関連」です。p 値は n が大きければいくらでも小さくなるので、関連の **強さ** は V で判断します。
- 2 × 2 の表で期待度数が小さいときは Fisher の正確検定 `stats.fisher_exact` を使います。

### 例題 35: 一元配置分散分析（ANOVA）と Tukey の HSD

**背景**: 3 種類の肥料 A, B, C をそれぞれ 12 区画の畑に使い、収量（kg）を測りました。肥料によって収量に差があるでしょうか。あるなら、どの肥料とどの肥料の間に差があるのでしょうか。

3 群以上の平均を比べるときに t 検定を繰り返すと、第 1 種の誤りが積み重なります（3 回で約 14%）。
まず **分散分析**（H0: すべての群の平均が等しい）で全体の差を調べ、有意なら **多重比較**（Tukey の HSD）でどのペアに差があるかを調べます。

In [ ]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd

rng = np.random.default_rng(34)
yields = {
    "肥料A": rng.normal(50, 5, size=12).round(1),
    "肥料B": rng.normal(58, 5, size=12).round(1),
    "肥料C": rng.normal(51, 5, size=12).round(1),
}
for name, y in yields.items():
    print(f"{name}: 平均 {y.mean():.2f} kg, SD {y.std(ddof=1):.2f}")

res = stats.f_oneway(*yields.values())
print(f"\n一元配置分散分析: F = {res.statistic:.3f}, p 値 = {res.pvalue:.4f}")

# 多重比較: Tukey の HSD
values = np.concatenate(list(yields.values()))
labels = np.repeat(list(yields.keys()), 12)
tukey = pairwise_tukeyhsd(endog=values, groups=labels, alpha=0.05)
print("\nTukey の HSD（reject = True のペアに有意差あり）:")
print(tukey)

plt.figure(figsize=(5, 3.5))
plt.boxplot(list(yields.values()))
plt.xticks([1, 2, 3], list(yields.keys()))
plt.ylabel("収量 (kg)")
plt.title("肥料ごとの収量")
plt.show()

**結果の読み方**

- 分散分析の p 値が 0.05 未満なら「少なくとも 1 つの群の平均が他と異なる」と言えますが、**どの群が違うかは分かりません**。
- Tukey の HSD の表は、各ペアについて平均差 (meandiff)、調整済み p 値 (p-adj)、差の信頼区間 (lower, upper)、有意かどうか (reject) を示します。この例では B と他の 2 つの間に差があり、A と C の間には差がありません。
- F 統計量は「群間のばらつき ÷ 群内のばらつき」です。群間の差が群内の散らばりに比べて大きいほど F が大きくなります。
- 分散分析は各群の正規性と等分散性を仮定します。等分散が疑わしいときは Welch の分散分析（`stats.alexandergovern`）や Kruskal-Wallis 検定を使います。

### 例題 36: 効果量（Cohen's d）

**背景**: 例題 28 では「差がある」と分かりましたが、その差は **どれくらい大きい** のでしょうか。また、n を大きくすると p 値と効果量はどう変わるでしょうか。

**Cohen's d** は 2 群の平均の差を、共通の標準偏差で割ったもので、単位に依存しない「差の大きさ」の指標です。

| d の目安 | 解釈 |
|---|---|
| 0.2 | 小さい |
| 0.5 | 中程度 |
| 0.8 | 大きい |

In [ ]:
def cohens_d(x, y):
    """2 群の平均の差をプールした標準偏差で割った効果量"""
    nx, ny = len(x), len(y)
    pooled_var = ((nx - 1) * np.var(x, ddof=1) + (ny - 1) * np.var(y, ddof=1)) / (nx + ny - 2)
    return (np.mean(y) - np.mean(x)) / np.sqrt(pooled_var)

d = cohens_d(group_a, group_b)
print(f"例題 28 のデータ (A vs B): Cohen's d = {d:.3f}")

# 真の差が 0.1 SD しかないデータで n を変えると…
rng = np.random.default_rng(135)
print("\n真の差 = 0.1 SD（ごく小さい効果）のとき:")
print(f"{'n (各群)':>10s} {'p 値':>10s} {'Cohen d':>10s}")
for n in [20, 200, 2000, 20000]:
    x = rng.normal(0, 1, size=n)
    y = rng.normal(0.1, 1, size=n)
    p = stats.ttest_ind(x, y).pvalue
    print(f"{n:10d} {p:10.4f} {cohens_d(x, y):10.3f}")

**結果の読み方**

- 例題 28 のデータでは d ≈ 1.6 と「非常に大きい」効果です。新方式の効果は統計的に有意なだけでなく、実質的にも大きいと言えます。
- 真の差が 0.1 SD しかない場合、n = 20 や 200 では有意になりませんが、n = 2000 以上では有意になり、n = 20000 では p 値が極めて小さくなります。一方、**効果量の推定値は n によらず 0.1 前後** で、n が大きいほど真の値 0.1 に近づくだけです（小さい n では推定のばらつきが大きい）。
- p 値は「差が 0 ではなさそうか」を、効果量は「差がどれくらい大きいか」を表します。ビッグデータの時代には、ほとんど何でも「有意」になるので、効果量と信頼区間で意味のある差かを判断してください。
- 対応のあるデータでは差の平均 ÷ 差の SD、相関では r そのもの、分散分析では η²（イータ二乗）が効果量として使われます。

### 例題 37: 並べ替え検定（permutation test）

**背景**: 新しい寝具を使った 8 人と従来の寝具を使った 8 人の平均睡眠時間（時間）を比べます。n が小さく正規分布の仮定にも自信がありません。

**データ**:
- 従来: `6.1, 7.0, 5.8, 6.5, 7.2, 6.8, 5.9, 6.3`
- 新寝具: `6.9, 6.4, 7.8, 6.6, 7.5, 6.2, 7.1, 6.8`

**並べ替え検定** の考え方: もし寝具に効果がない（H0）なら、16 個のデータのどれが「新寝具」ラベルでも同じはずです。ラベルを無作為にシャッフルして平均の差を計算する作業を 1 万回繰り返し、実際に観測された差以上の差が出る割合を p 値とします。分布の仮定が一切いらない検定です。

In [ ]:
rng = np.random.default_rng(36)
old = np.array([6.1, 7.0, 5.8, 6.5, 7.2, 6.8, 5.9, 6.3])
new = np.array([6.9, 6.4, 7.8, 6.6, 7.5, 6.2, 7.1, 6.8])
observed_diff = new.mean() - old.mean()

pooled = np.concatenate([old, new])
n_old = len(old)
n_perm = 10000
perm_diffs = np.empty(n_perm)
for i in range(n_perm):
    shuffled = rng.permutation(pooled)                # ラベルをシャッフル
    perm_diffs[i] = shuffled[n_old:].mean() - shuffled[:n_old].mean()

p_perm = np.mean(np.abs(perm_diffs) >= abs(observed_diff))   # 両側
print(f"観測された平均の差 (新 - 旧): {observed_diff:.3f} 時間")
print(f"並べ替え検定の p 値: {p_perm:.4f}  （{n_perm} 回中 {int(round(p_perm * n_perm))} 回が観測値以上の差）")
print(f"（参考）Welch の t 検定の p 値: {stats.ttest_ind(new, old, equal_var=False).pvalue:.4f}")
print(f"（参考）Mann-Whitney の p 値  : {stats.mannwhitneyu(new, old).pvalue:.4f}")

plt.figure(figsize=(6, 3.8))
plt.hist(perm_diffs, bins=50, color="lightgray", edgecolor="white")
plt.axvline(observed_diff, color="red", label=f"観測値 {observed_diff:+.3f}")
plt.axvline(-observed_diff, color="red", linestyle="--")
plt.xlabel("平均の差（H0 のもとでの分布）")
plt.ylabel("度数")
plt.title("並べ替え検定: シャッフルで得られる差の分布")
plt.legend()
plt.show()

**結果の読み方**

- 観測された差は約 0.46 時間ですが、ラベルをシャッフルしただけでもそれ以上の差がおよそ 10% の確率で出ます。p ≈ 0.1 で有意水準 5% では棄却されません。
- ヒストグラムは「効果がない世界で偶然生じる差の分布」です。観測値（赤線）がこの分布の端の方にあるほど p 値は小さくなります。
- t 検定の p 値とほぼ同じになりました。データが正規分布に近ければ両者はよく一致し、そうでなければ並べ替え検定の方が信頼できます。
- シャッフル回数を増やせば p 値の精度は上がります（`scipy.stats.permutation_test` には同じ機能があります）。

### 例題 38: 検出力とサンプルサイズ

**背景**: 実際には 0.3 SD の差がある 2 群を比べるとき、各群 n 人の t 検定で「有意」と判定できる確率（**検出力**）はどれくらいでしょうか。検出力 80% を確保するには何人必要でしょうか。

1 群あたりのサンプルサイズを変えて、400 回ずつ実験（乱数で 2 群を作って t 検定）し、p < 0.05 になった割合を数えます。statsmodels の理論値とも比べます。

In [ ]:
from statsmodels.stats.power import TTestIndPower

rng = np.random.default_rng(37)
effect = 0.3            # 真の差 = 0.3 SD
n_list = [10, 20, 50, 100, 200, 400]
n_sim = 400
power_sim = []
for n in n_list:
    x = rng.normal(0, 1, size=(n_sim, n))
    y = rng.normal(effect, 1, size=(n_sim, n))
    p = stats.ttest_ind(x, y, axis=1).pvalue      # 400 回分の検定を一度に計算
    power_sim.append(np.mean(p < 0.05))

analysis = TTestIndPower()
power_theory = [analysis.power(effect_size=effect, nobs1=n, alpha=0.05) for n in n_list]
print(f"{'n (各群)':>9s} {'検出力(シミュレーション)':>18s} {'検出力(理論値)':>12s}")
for n, ps, pt in zip(n_list, power_sim, power_theory):
    print(f"{n:9d} {ps:18.3f} {pt:12.3f}")

n_needed = analysis.solve_power(effect_size=effect, alpha=0.05, power=0.8)
print(f"\nd = {effect} を検出力 80% で検出するのに必要なサンプルサイズ: 各群 {np.ceil(n_needed):.0f} 人")
for d in [0.2, 0.5, 0.8]:
    print(f"  d = {d}: 各群 {np.ceil(analysis.solve_power(effect_size=d, alpha=0.05, power=0.8)):.0f} 人")

plt.figure(figsize=(6, 3.8))
plt.plot(n_list, power_sim, "o-", label="シミュレーション")
plt.plot(n_list, power_theory, "s--", label="理論値")
plt.axhline(0.8, color="gray", linestyle=":", label="検出力 0.8")
plt.xscale("log")
plt.xlabel("1 群あたりのサンプルサイズ（対数目盛）")
plt.ylabel("検出力（p < 0.05 となる割合）")
plt.title("効果量 d = 0.3 のときの検出力")
plt.legend()
plt.show()

**結果の読み方**

- 各群 10 人では検出力は 10% 程度しかなく、本当に差があっても 10 回に 9 回は「有意でない」という結果になります。各群 175 人前後でようやく 80% に達します。
- 「有意にならなかった」研究の多くは、差がないのではなく **サンプルサイズが足りなかった** だけかもしれません。p ≥ 0.05 を「差がない証拠」と解釈してはいけない理由がここにあります。
- 効果量が小さいほど必要な n は急増します（d = 0.2 なら各群約 400 人、d = 0.8 なら約 26 人）。
- 実験の前に「期待される効果量」「α」「目標の検出力」から n を決める作業を **検出力分析** と言います。`TTestIndPower().solve_power` はそのための道具です。

## 7. 相関と回帰

### 例題 39: ピアソンの相関係数とその検定

**背景**: 25 人の学生について、1 週間の勉強時間とテストの点数を記録しました。勉強時間が長い学生ほど点数が高いと言えるでしょうか。

**ピアソンの相関係数 r** は 2 つの変数の **直線的な関係** の強さを −1〜1 で表します。`stats.pearsonr` は r と「母集団の相関が 0 である」という帰無仮説の p 値、さらに r の信頼区間を返します。

In [ ]:
rng = np.random.default_rng(38)
n = 25
study_hours = rng.uniform(0, 10, size=n).round(1)
score = (50 + 4 * study_hours + rng.normal(0, 8, size=n)).round(0)

res = stats.pearsonr(study_hours, score)
ci = res.confidence_interval(0.95)
print(f"ピアソンの相関係数 r = {res.statistic:.3f}, p 値 = {res.pvalue:.4g}")
print(f"r の 95% 信頼区間: [{ci.low:.3f}, {ci.high:.3f}]")
print(f"決定係数 r² = {res.statistic ** 2:.3f}  → 点数のばらつきの {res.statistic ** 2:.0%} が勉強時間で説明できる")

# 手計算で確認: r = 共分散 / (SD_x × SD_y)
cov = np.cov(study_hours, score, ddof=1)[0, 1]
print(f"手計算: 共分散 {cov:.2f} / ({study_hours.std(ddof=1):.3f} × {score.std(ddof=1):.3f}) = {cov / (study_hours.std(ddof=1) * score.std(ddof=1)):.3f}")

plt.figure(figsize=(5.5, 3.8))
plt.scatter(study_hours, score, color="steelblue")
plt.xlabel("1 週間の勉強時間 (時間)")
plt.ylabel("テストの点数")
plt.title(f"勉強時間と点数 (r = {res.statistic:.2f})")
plt.show()

**結果の読み方**

- r ≈ 0.8 の強い正の相関で、p 値は非常に小さく「相関がない」という仮説は棄却されます。
- 相関係数の信頼区間は n = 25 だとかなり広いことに注意してください。r の点推定値だけで「0.8 の相関がある」と断定するのは危険です。
- 「相関の検定で有意」は「r ≠ 0 と言える」というだけで、相関が強いことを意味しません。n が大きければ r = 0.05 でも有意になります。
- 相関係数は **外れ値に非常に弱い** です。必ず散布図で確認しましょう。

### 例題 40: スピアマンの順位相関

**背景**:

1. x と y の関係が「単調に増えるが直線ではない」（指数関数的に増える）とき、ピアソンとスピアマンの相関係数はどう違うでしょうか。
2. 2 人の審査員が 10 作品につけた順位はどれくらい一致しているでしょうか。

**スピアマンの順位相関係数 ρ** は、値を順位に置き換えてからピアソンの相関を計算したものです。直線でなくても単調な関係なら高い値になり、外れ値の影響も受けにくく、順位データ（順序尺度）にも使えます。

In [ ]:
rng = np.random.default_rng(39)
x = np.linspace(1, 10, 30)
y = np.exp(0.5 * x) + rng.normal(0, 5, size=30)     # 単調だが直線的ではない関係

r_p, p_p = stats.pearsonr(x, y)
r_s, p_s = stats.spearmanr(x, y)
print(f"ピアソン   r = {r_p:.3f} (p = {p_p:.2g})")
print(f"スピアマン ρ = {r_s:.3f} (p = {p_s:.2g})")

# 順位データ: 2 人の審査員が 10 作品につけた順位
judge1 = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
judge2 = [2, 1, 4, 3, 6, 5, 7, 9, 8, 10]
rho, p = stats.spearmanr(judge1, judge2)
tau, p_tau = stats.kendalltau(judge1, judge2)
print(f"\n審査員の順位の一致度: スピアマン ρ = {rho:.3f} (p = {p:.4f}), ケンドール τ = {tau:.3f} (p = {p_tau:.4f})")

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
axes[0].scatter(x, y, color="darkorange")
axes[0].set_title(f"元の値（ピアソン r = {r_p:.2f}）")
axes[0].set_xlabel("x")
axes[0].set_ylabel("y")
axes[1].scatter(stats.rankdata(x), stats.rankdata(y), color="darkgreen")
axes[1].set_title(f"順位に変換（スピアマン ρ = {r_s:.2f}）")
axes[1].set_xlabel("x の順位")
axes[1].set_ylabel("y の順位")
plt.tight_layout()
plt.show()

**結果の読み方**

- 指数的な関係では、ピアソンの r は 0.9 前後にとどまりますが、スピアマンの ρ は約 0.95 まで上がります。順位に変換すると右のグラフのようにきれいな直線になるためです。
- 審査員の順位の ρ ≈ 0.95 は「2 人の評価はよく一致している」ことを示します。ケンドールの τ も同様の指標で、値は ρ より小さめに出るのが普通です。
- 使い分けの目安: 両方とも量的データで直線的な関係 → ピアソン。順位データ、単調だが非線形、外れ値がある → スピアマン。
- どちらの相関係数も「単調な関係」しか捉えません。U 字型のような関係は相関係数がほぼ 0 になるので、やはり散布図が必要です。

### 例題 41: 相関 ≠ 因果（交絡変数のシミュレーション）

**背景**: 「アイスクリームの売上が多い日は水難事故が多い」という相関がよく知られています。アイスを売るのをやめれば事故は減るでしょうか。

もちろん違います。**気温** が高い日はアイスも売れるし泳ぐ人も増える、という **交絡変数** が両方に影響しているだけです。
乱数でこの状況を再現し、気温の影響を取り除いた **偏相関** を計算してみます。

In [ ]:
rng = np.random.default_rng(240)
days = 120
temp = rng.uniform(10, 35, size=days)                       # 気温（交絡変数）
ice_sales = 20 + 3 * temp + rng.normal(0, 8, size=days)     # アイスの売上: 気温で決まる
drownings = 0.2 * temp + rng.normal(0, 1.2, size=days)      # 水難事故件数: 気温で決まる（架空のデータ）

r_raw = stats.pearsonr(ice_sales, drownings)
print(f"アイスの売上 と 水難事故 の相関: r = {r_raw.statistic:.3f}, p = {r_raw.pvalue:.2g}")

# 偏相関: それぞれを気温で回帰した残差同士の相関
resid_ice = ice_sales - np.polyval(np.polyfit(temp, ice_sales, 1), temp)
resid_drown = drownings - np.polyval(np.polyfit(temp, drownings, 1), temp)
r_partial = stats.pearsonr(resid_ice, resid_drown)
print(f"気温の影響を取り除いた偏相関: r = {r_partial.statistic:.3f}, p = {r_partial.pvalue:.2f}")

fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.5))
sc = axes[0].scatter(ice_sales, drownings, c=temp, cmap="coolwarm", s=18)
axes[0].set_xlabel("アイスの売上")
axes[0].set_ylabel("水難事故件数")
axes[0].set_title("見かけの相関（色 = 気温）")
fig.colorbar(sc, ax=axes[0], label="気温 (°C)")
axes[1].scatter(resid_ice, resid_drown, s=18, color="gray")
axes[1].set_xlabel("アイスの売上（気温の影響を除去）")
axes[1].set_ylabel("水難事故（気温の影響を除去）")
axes[1].set_title("交絡を取り除くと相関は消える")
plt.tight_layout()
plt.show()

**結果の読み方**

- アイスの売上と水難事故には r ≈ 0.74 の強い相関がありますが、気温を調整した偏相関は約 −0.06 で、有意でもありません（p ≈ 0.55）。相関は気温という第 3 の変数が作り出した **見かけの相関** でした。
- 観察データから因果関係を主張するには、交絡変数を測定して調整する（重回帰など）か、無作為化実験（A/B テスト、ランダム化比較試験）が必要です。
- 相関が因果でない典型的なパターン: (1) 交絡（この例）、(2) 逆の因果（売上が増えたから広告を出した）、(3) 偶然（n が小さいときに特に起こる）。
- 「相関があるからといって因果があるとは限らない」は、統計を学ぶうえで最も大切な教訓の一つです。

### 例題 42: 単回帰分析（`stats.linregress`）

**背景**: 12 か月分の広告費（万円）と売上（万円）のデータがあります。広告費が 1 万円増えると売上はいくら増えるでしょうか。広告費 80 万円のときの売上を予測してください。

**データ**:
- 広告費: `10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 70`
- 売上: `120, 135, 150, 170, 175, 200, 205, 230, 240, 250, 275, 300`

単回帰は「y = 切片 + 傾き × x」という直線をデータに当てはめます。`stats.linregress` は傾き・切片・相関係数・p 値・傾きの標準誤差を一度に返します。

In [ ]:
ad_cost = np.array([10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 70])
sales_amt = np.array([120, 135, 150, 170, 175, 200, 205, 230, 240, 250, 275, 300])

res = stats.linregress(ad_cost, sales_amt)
print(f"回帰式: 売上 = {res.intercept:.2f} + {res.slope:.3f} × 広告費")
print(f"傾きの標準誤差 = {res.stderr:.3f}, 傾きの p 値 = {res.pvalue:.2g}")
t_crit = stats.t.ppf(0.975, df=len(ad_cost) - 2)
print(f"傾きの 95% 信頼区間: [{res.slope - t_crit * res.stderr:.3f}, {res.slope + t_crit * res.stderr:.3f}]")
print(f"相関係数 r = {res.rvalue:.3f}, 決定係数 R² = {res.rvalue ** 2:.3f}")

predicted = res.intercept + res.slope * ad_cost
residuals = sales_amt - predicted
print(f"残差の標準偏差（予測誤差の目安）: {np.sqrt(np.sum(residuals ** 2) / (len(ad_cost) - 2)):.2f} 万円")
new_cost = 80
print(f"\n広告費 {new_cost} 万円のときの予測売上: {res.intercept + res.slope * new_cost:.1f} 万円（データの範囲外への外挿なので注意）")

plt.figure(figsize=(5.5, 3.8))
plt.scatter(ad_cost, sales_amt, color="steelblue", label="観測値")
xs = np.linspace(0, 85, 50)
plt.plot(xs, res.intercept + res.slope * xs, color="red", label="回帰直線")
plt.scatter([new_cost], [res.intercept + res.slope * new_cost], color="red", marker="x", s=80, label="予測 (80 万円)")
plt.xlabel("広告費 (万円)")
plt.ylabel("売上 (万円)")
plt.title(f"単回帰 (R² = {res.rvalue ** 2:.3f})")
plt.legend()
plt.show()

**結果の読み方**

- 傾きは約 3.0 で「広告費が 1 万円増えると売上は約 3 万円増える」、切片は約 90 で「広告費 0 のときの売上は約 90 万円」と読みます（切片は x = 0 が現実的な場合のみ意味があります）。
- 傾きの p 値が小さいのは「傾きが 0 ではない」= 広告費と売上に直線的な関係がある、ということです。
- R² ≈ 0.99 は売上のばらつきの 99% が広告費で説明できることを意味します。実際のビジネスデータでここまで高い値はまれです。
- 広告費 80 万円はデータの範囲（10〜70）の外側です。**外挿** は関係が範囲外でも直線のまま続くという仮定に依存するので、信頼性が落ちます。
- 回帰は「x から y を予測する」と「y から x を予測する」で別の直線になります。相関係数と違って対称ではありません。

### 例題 43: 重回帰分析（statsmodels の `ols` と summary の読み方）

**背景**: 中古マンション 80 件の価格（万円）を、専有面積（m²）、築年数、駅までの徒歩（分）から説明したい。それぞれの要因は価格にどれくらい影響しているでしょうか。

説明変数が複数ある回帰を **重回帰** と言います。statsmodels の `ols` は R 風の **formula**（`"目的変数 ~ 説明変数1 + 説明変数2"`）でモデルを書け、`summary()` で詳細な結果表が得られます。

In [ ]:
import statsmodels.api as sm
from statsmodels.formula.api import ols

rng = np.random.default_rng(42)
n = 80
house = pd.DataFrame({
    "area": rng.uniform(40, 120, size=n).round(1),     # 専有面積 (m²)
    "age": rng.integers(0, 40, size=n),                # 築年数
    "dist": rng.uniform(1, 20, size=n).round(1),       # 駅まで徒歩 (分)
})
# 真の関係: 価格 = 500 + 40×面積 − 25×築年数 − 30×徒歩分 + ノイズ
house["price"] = (500 + 40 * house["area"] - 25 * house["age"] - 30 * house["dist"]
                  + rng.normal(0, 300, size=n)).round(0)
print(house.head())

model = ols("price ~ area + age + dist", data=house).fit()
print(model.summary())

**結果の読み方（summary の主な項目）**

| 項目 | 意味 |
|---|---|
| `R-squared` / `Adj. R-squared` | 決定係数 / 自由度調整済み決定係数。説明変数を増やすと R² は必ず上がるので、モデル比較には Adj. R² を使う |
| `F-statistic`, `Prob (F-statistic)` | 「すべての係数が 0」という帰無仮説の検定。p が小さければモデル全体に意味がある |
| `coef` | 各説明変数の係数。**他の変数を固定したときに**、その変数が 1 増えると目的変数がいくつ変わるか |
| `std err`, `t`, `P>|t|` | 係数の標準誤差、t 値、p 値。p < 0.05 ならその変数は有意 |
| `[0.025  0.975]` | 係数の 95% 信頼区間 |
| `AIC`, `BIC` | モデル選択の指標（小さいほど良い） |
| `Durbin-Watson` | 残差の自己相関（2 に近ければ問題なし） |
| `Cond. No.` | 多重共線性の目安。数百〜数千以上なら説明変数同士の相関を疑う |

- area, age, dist の係数は真の値（40, −25, −30）に近く、いずれも p < 0.001 で有意です。切片は p ≈ 0.08 で有意ではありませんが、切片の検定（「面積 0 m²・築 0 年・駅から 0 分の価格が 0 か」）は通常あまり意味がありません。「面積が 1 m² 広いと約 40 万円高い」「築 1 年ごとに約 25 万円安い」「駅から 1 分遠いごとに約 30 万円安い」と読めます。
- 係数の大きさは変数の単位に依存するので、係数を直接比べて「どの変数が最も重要か」とは言えません。比べたいときは説明変数を標準化してから回帰します。

### 例題 44: 回帰モデルの係数の取り出し、予測、残差診断

**背景**: 例題 43 のモデルから必要な数値だけを取り出し、新しい物件の価格を予測します。また、回帰の前提（残差が平均 0・等分散・正規分布）が満たされているかを **残差プロット** で確認します。

In [ ]:
coef = pd.DataFrame({"係数": model.params, "標準誤差": model.bse, "p値": model.pvalues})
ci = model.conf_int()
coef["CI下限"] = ci[0]
coef["CI上限"] = ci[1]
print(coef.round(3))
print(f"\nR² = {model.rsquared:.3f}, 調整済み R² = {model.rsquared_adj:.3f}, 残差の標準誤差 = {np.sqrt(model.mse_resid):.1f} 万円")

new_house = pd.DataFrame({"area": [70, 100], "age": [5, 30], "dist": [8, 15]})
pred = model.get_prediction(new_house).summary_frame(alpha=0.05)
print("\n新しい物件の予測（mean = 予測価格, obs_ci = 個々の物件の 95% 予測区間）:")
print(pd.concat([new_house, pred[["mean", "obs_ci_lower", "obs_ci_upper"]].round(0)], axis=1))

fig, axes = plt.subplots(1, 3, figsize=(11, 3.3))
axes[0].scatter(model.fittedvalues, model.resid, s=18, color="steelblue")
axes[0].axhline(0, color="red", linestyle="--")
axes[0].set_xlabel("予測値")
axes[0].set_ylabel("残差")
axes[0].set_title("残差 vs 予測値")
axes[1].hist(model.resid, bins=15, color="steelblue", edgecolor="white")
axes[1].set_xlabel("残差")
axes[1].set_title("残差のヒストグラム")
stats.probplot(model.resid, dist="norm", plot=axes[2])
axes[2].set_title("残差の Q-Q プロット")
axes[2].set_xlabel("理論分位点")
axes[2].set_ylabel("残差の分位点")
plt.tight_layout()
plt.show()

**結果の読み方**

- `model.params`、`model.pvalues`、`model.conf_int()` などで summary の中身を個別に取り出せます。レポート用の表はこのように DataFrame にまとめると便利です。
- 予測区間（obs_ci）は信頼区間より広いことに注意してください。「平均的な物件の価格」ではなく「この 1 件の物件の価格」の不確かさを表すので、個々のばらつき（残差）の分も含まれます。
- 残差 vs 予測値のプロットは、残差が 0 を中心にランダムに散らばっていれば OK です。次のパターンが見えたら要注意:
  - 曲線状のパターン → 非線形の関係を見逃している（2 乗項や対数変換を検討）
  - 予測値が大きいほど散らばりが大きい（扇形） → 不等分散（対数変換や頑健な標準誤差を検討）
- 残差のヒストグラムと Q-Q プロットが正規分布から大きく外れていなければ、係数の p 値や信頼区間は信頼できます。

### 例題 45: iris で種別ごとの回帰（シンプソンのパラドックス）

**背景**: iris の全データで「がく片の長さ (sepal_length) → がく片の幅 (sepal_width)」の回帰をすると、傾きは負になります。ところが種ごとに回帰すると傾きはすべて正です。どちらが正しいのでしょうか。

全体で見た関係とグループ内の関係が逆転する現象を **シンプソンのパラドックス** と言います。
回帰式に `C(species)` を加えると、種の違いを調整した傾きが得られます。

In [ ]:
res_all = stats.linregress(iris["sepal_length"], iris["sepal_width"])
print(f"全体      : 傾き {res_all.slope:+.3f}, r = {res_all.rvalue:+.3f}, p = {res_all.pvalue:.3f}")

colors = {"setosa": "tab:blue", "versicolor": "tab:orange", "virginica": "tab:green"}
plt.figure(figsize=(6, 4.2))
xs = np.linspace(4.2, 8, 50)
plt.plot(xs, res_all.intercept + res_all.slope * xs, color="black", linestyle="--", label="全体の回帰直線")
for sp, color in colors.items():
    sub = iris[iris["species"] == sp]
    r = stats.linregress(sub["sepal_length"], sub["sepal_width"])
    print(f"{sp:10s}: 傾き {r.slope:+.3f}, r = {r.rvalue:+.3f}, p = {r.pvalue:.2g}")
    plt.scatter(sub["sepal_length"], sub["sepal_width"], s=16, color=color, alpha=0.7, label=sp)
    xr = np.linspace(sub["sepal_length"].min(), sub["sepal_length"].max(), 20)
    plt.plot(xr, r.intercept + r.slope * xr, color=color)
plt.xlabel("がく片の長さ (cm)")
plt.ylabel("がく片の幅 (cm)")
plt.title("全体では負の傾き、種ごとでは正の傾き")
plt.legend(fontsize=8)
plt.show()

# 種の違いを調整した重回帰（種をダミー変数として加える）
model_sp = ols("sepal_width ~ sepal_length + C(species)", data=iris).fit()
print("\n種を調整した回帰の係数:")
print(model_sp.params.round(3))
print(f"sepal_length の p 値 = {model_sp.pvalues['sepal_length']:.2g}, R² = {model_sp.rsquared:.3f}")

**結果の読み方**

- 全体の回帰では傾きが負（r ≈ −0.11）ですが、種ごとに見るとすべて正の傾きで、setosa では r ≈ 0.74 の強い正の相関があります。
- 原因は、setosa が「がく片が短くて幅が広い」、virginica が「長くて幅が狭め」という **種の違い** です。種をまたいで一直線に当てはめると、この群間の違いが負の傾きとして現れます。
- 種を説明変数に加えた重回帰では sepal_length の係数は正（約 0.35）になり、有意です。これが「種の違いを調整したうえでの、長さと幅の関係」です。
- グループ構造のあるデータでは、必ず **グループ別に色分けした散布図** を描き、全体の相関を鵜呑みにしないようにしましょう。

### 例題 46: ロジスティック回帰（2 値の分類）

**背景**: iris の versicolor と virginica を花弁の長さ (petal_length) から見分けたい。目的変数が「virginica かどうか（0 / 1）」のような 2 値のときは、線形回帰ではなく **ロジスティック回帰** を使います。

ロジスティック回帰は「virginica である確率」を次の式でモデル化します。

$$P(y = 1) = \frac{1}{1 + e^{-(b_0 + b_1 x)}}$$

係数 b1 の指数 exp(b1) は **オッズ比**（x が 1 増えたときにオッズが何倍になるか）として解釈します。

In [ ]:
from statsmodels.formula.api import logit

two = iris[iris["species"].isin(["versicolor", "virginica"])].copy()
two["is_virginica"] = (two["species"] == "virginica").astype(int)

model_l = logit("is_virginica ~ petal_length", data=two).fit(disp=0)
print(model_l.summary().tables[1])        # 係数の表だけ表示
b0, b1 = model_l.params["Intercept"], model_l.params["petal_length"]
print(f"\nオッズ比 exp(b1) = {np.exp(b1):.2f}  → 花弁が 1 cm 長くなるごとに virginica のオッズが約 {np.exp(b1):.0f} 倍")
print(f"確率が 50% になる花弁の長さ = -b0/b1 = {-b0 / b1:.2f} cm")
print(f"擬似決定係数 (McFadden) = {model_l.prsquared:.3f}")

new = pd.DataFrame({"petal_length": [4.0, 4.5, 4.9, 5.0, 5.5, 6.0]})
new["P(virginica)"] = model_l.predict(new)
print("\n花弁の長さごとの予測確率:")
print(new.round(3).to_string(index=False))

pred_class = (model_l.predict(two) >= 0.5).astype(int)
acc = (pred_class == two["is_virginica"]).mean()
print(f"\n正解率（しきい値 0.5）: {acc:.3f}")
print("混同行列（行 = 実際, 列 = 予測）:")
print(pd.crosstab(two["is_virginica"], pred_class, rownames=["実際"], colnames=["予測"]))

xs = np.linspace(2.5, 7.5, 200)
plt.figure(figsize=(6, 3.8))
plt.scatter(two["petal_length"], two["is_virginica"], s=16, alpha=0.5, color="steelblue", label="観測値 (0 = versicolor, 1 = virginica)")
plt.plot(xs, model_l.predict(pd.DataFrame({"petal_length": xs})), color="red", label="予測確率")
plt.axhline(0.5, color="gray", linestyle=":")
plt.xlabel("花弁の長さ (cm)")
plt.ylabel("P(virginica)")
plt.title("ロジスティック回帰")
plt.legend(fontsize=8)
plt.show()

**結果の読み方**

- 係数 b1 は正で有意です。花弁が長いほど virginica である確率が高くなります。オッズ比は非常に大きく、1 cm の違いで確率が劇的に変わることを意味します。
- 予測確率は S 字カーブ（シグモイド）を描き、境界（約 4.9 cm）を境に 0 から 1 へ急に切り替わります。
- 正解率は約 93% ですが、正解率だけでは不十分なことも多いです。混同行列で「どちらの誤りが多いか」を確認しましょう。医療診断のように見逃しのコストが高い場面では、しきい値を 0.5 から動かすことも検討します。
- ここでは学習に使ったデータで正解率を評価しているので、楽観的な値になっています。本来は学習用と評価用にデータを分けて評価します（機械学習の入門で学ぶ内容です）。

## 8. 時系列の基礎

時間順に並んだデータ（時系列）は、隣り合う値が互いに関係しているため、これまでの「独立な観測」を前提とした手法をそのまま使えないことがあります。ここでは時系列を眺めるための基本的な道具を 3 つ紹介します。

### 例題 47: 移動平均でノイズをならす

**背景**: Web サイトの日次訪問者数 120 日分があります。日ごとのノイズや週末の落ち込みが大きくて傾向が見にくいので、**移動平均** でならしてトレンドを見たい。

移動平均は「その日を中心とする直近 k 日の平均」です。pandas の `rolling(window=k).mean()` で計算できます。

In [ ]:
rng = np.random.default_rng(46)
dates = pd.date_range("2024-01-01", periods=120, freq="D")
trend = np.linspace(100, 160, 120)                        # ゆるやかな増加
weekly = np.where(dates.dayofweek >= 5, -30, 10)          # 週末は少ない
visits = pd.Series(trend + weekly + rng.normal(0, 12, size=120), index=dates, name="visits").round(0)

ma7 = visits.rolling(window=7, center=True).mean()       # 7 日移動平均（週の周期を消す）
ma28 = visits.rolling(window=28, center=True).mean()     # 28 日移動平均（より滑らか）
print(pd.DataFrame({"訪問者数": visits, "7日移動平均": ma7, "28日移動平均": ma28}).iloc[12:19].round(1))

plt.figure(figsize=(7, 3.8))
plt.plot(visits.index, visits, color="lightgray", label="日次訪問者数")
plt.plot(ma7.index, ma7, color="steelblue", label="7 日移動平均")
plt.plot(ma28.index, ma28, color="red", label="28 日移動平均")
plt.xlabel("日付")
plt.ylabel("訪問者数")
plt.title("移動平均でノイズと週の周期をならす")
plt.legend()
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

**結果の読み方**

- 元データ（灰色）は週末の落ち込みとノイズでギザギザですが、7 日移動平均（青）では週の周期がほぼ消え、28 日移動平均（赤）ではなだらかな増加トレンドだけが残ります。
- 週の周期をきれいに消すには、窓の長さを周期の整数倍（7, 14, 28 …）にするのがコツです。
- 窓を長くするほど滑らかになりますが、急な変化への反応が鈍くなり、両端で値が計算できない期間（NaN）も長くなります。`center=True` を付けないと、移動平均は過去の値だけで計算され、グラフが右にずれます。
- 移動平均のほかに、直近の値を重視する **指数平滑化**（`ewm().mean()`）もよく使われます。

### 例題 48: 差分と前年同月比

**背景**: 4 年分（48 か月）の月次売上があります。売上は毎年少しずつ成長しつつ、夏と 12 月に多い季節性があります。「先月より増えたか」「去年の同じ月より増えたか」を計算したい。

- **差分** `diff()`: 前の期との差。
- **変化率**: 前の期と比べた増減率（%）。
- **前年同月比**: 12 期前と比べた変化率。季節性の影響を受けずに成長を見られる。

In [ ]:
rng = np.random.default_rng(47)
months = pd.date_range("2021-01-01", periods=48, freq="MS")
season = np.array([0.8, 0.8, 0.9, 1.0, 1.0, 1.1, 1.3, 1.4, 1.1, 1.0, 0.9, 1.2])   # 夏と 12 月に多い
base = 100 * (1.006 ** np.arange(48))                                            # 月 0.6% ずつ成長
monthly = pd.Series((base * np.tile(season, 4) * rng.normal(1, 0.04, size=48)).round(1),
                    index=months, name="sales")

ts = pd.DataFrame({"売上": monthly})
ts["前月差"] = ts["売上"].diff()
ts["前月比(%)"] = (ts["売上"] / ts["売上"].shift(1) - 1) * 100
ts["前年同月比(%)"] = (ts["売上"] / ts["売上"].shift(12) - 1) * 100
print(ts.tail(8).round(1))
print(f"\n前月比の平均 {ts['前月比(%)'].mean():+.2f}%, 前年同月比の平均 {ts['前年同月比(%)'].mean():+.2f}%")

yoy = ts["前年同月比(%)"].dropna()
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(ts.index, ts["売上"], marker=".")
axes[0].set_title("月次売上（成長 + 季節性）")
axes[0].set_ylabel("売上")
axes[1].bar(yoy.index, yoy.values, width=20, color=np.where(yoy.values >= 0, "steelblue", "red"))
axes[1].axhline(0, color="black", lw=0.8)
axes[1].set_title("前年同月比 (%)")
axes[1].set_ylabel("%")
for ax in axes:
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

**結果の読み方**

- 前月比は季節性のせいで大きく上下し（例: 9 月は 8 月より大きく減る）、成長しているのかどうか分かりません。
- 前年同月比は季節性を打ち消すので、ノイズで上下はするもののすべて正の値（平均 約 +9%）に収まり、「年率 7〜10% 程度で成長している」と読み取れます（設定した月 0.6% の成長は年率にすると 1.006¹² ≈ 1.074 です）。
- `shift(12)` は 12 行前の値を参照するので、最初の 12 か月は前年同月比が NaN になります。
- 前年同月比は季節性を扱う実務的な方法ですが、前年が異常（例: 特需や災害）だと比較がゆがみます。2 年前とも比べる、次の例題の季節分解を使う、といった工夫が必要です。

### 例題 49: 季節分解（`seasonal_decompose`）

**背景**: 例題 48 の月次売上を、**トレンド**（長期的な増減）、**季節成分**（毎年繰り返すパターン）、**残差**（それ以外）の 3 つに分解して、それぞれの大きさを確かめたい。

statsmodels の `seasonal_decompose` は移動平均でトレンドを求め、残りから季節パターンを平均して取り出します。季節性の振幅が売上の水準に比例するデータなので、乗法モデル（元データ = トレンド × 季節 × 残差）を使います。

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

decomp = seasonal_decompose(monthly, model="multiplicative", period=12)

fig, axes = plt.subplots(4, 1, figsize=(7, 7), sharex=True)
axes[0].plot(monthly.index, monthly)
axes[0].set_ylabel("元データ")
axes[0].set_title("季節分解（乗法モデル）: 元データ = トレンド × 季節 × 残差")
axes[1].plot(decomp.trend.index, decomp.trend, color="red")
axes[1].set_ylabel("トレンド")
axes[2].plot(decomp.seasonal.index, decomp.seasonal, color="green")
axes[2].set_ylabel("季節成分")
axes[3].scatter(decomp.resid.index, decomp.resid, s=10, color="gray")
axes[3].axhline(1, color="black", lw=0.8)
axes[3].set_ylabel("残差")
plt.tight_layout()
plt.show()

seasonal_factor = pd.Series(decomp.seasonal.iloc[:12].values, index=range(1, 13), name="季節係数")
print("月ごとの季節係数（1 より大きい月は平均的な月より売上が多い）:")
print(seasonal_factor.round(3).to_string())
print(f"\n残差の標準偏差: {decomp.resid.std():.3f}  （1 ± この程度の説明できない変動が残る）")

**結果の読み方**

- トレンドはほぼ一定の割合で増え、季節成分は 7〜8 月に約 1.25〜1.35、12 月に約 1.17、1〜2 月に約 0.77〜0.79 と、データを作ったときの季節パターンをほぼ復元しています。
- 残差は 1 の周りに ±0.03 程度でランダムに散らばっています。残差にパターンが残っていれば、周期の指定（`period`）やモデル（加法 / 乗法）が合っていない可能性があります。
- 加法モデル（`model="additive"`）は季節変動の振幅が水準に関係なく一定のとき、乗法モデルは水準に比例して大きくなるときに使います。
- トレンドは移動平均で求めるので、両端の 6 か月は NaN になります。季節調整済みの値は `monthly / decomp.seasonal` で得られ、前年同月比と同様に季節性を除いて比較できます。
- より本格的な時系列分析（ARIMA、指数平滑法、予測）は `statsmodels.tsa` に用意されています。

## 9. 練習問題

ここまでの内容を使って解いてみましょう。各問題について、(1) どの手法を使うべきか、(2) 結果をどう解釈するかを考えてください。
自分で書いてから、解答例と比べてみましょう。

### 問題 1: 2 店舗の売上を比べる

2 つの店舗の 1 日あたりの売上（万円）を 12 日ずつ記録しました。店舗 B の方が売上が多いと言えるでしょうか。適切な検定を選んで実行し、効果量も求めてください。

- 店舗 A: `52, 48, 61, 55, 47, 58, 50, 53, 49, 56, 60, 45`
- 店舗 B: `58, 63, 55, 66, 60, 57, 69, 62, 59, 64, 61, 67`

In [ ]:
# 問題 1: ここにコードを書いてください

**解答例**

独立した 2 群の平均の比較なので **Welch の t 検定**（`stats.ttest_ind(..., equal_var=False)`）を使います。n が小さいので正規性も確認しておきます。

In [ ]:
store_a = np.array([52, 48, 61, 55, 47, 58, 50, 53, 49, 56, 60, 45])
store_b = np.array([58, 63, 55, 66, 60, 57, 69, 62, 59, 64, 61, 67])

print(f"A: 平均 {store_a.mean():.2f}, SD {store_a.std(ddof=1):.2f}")
print(f"B: 平均 {store_b.mean():.2f}, SD {store_b.std(ddof=1):.2f}")
print(f"正規性 (Shapiro-Wilk) A: p = {stats.shapiro(store_a).pvalue:.3f}, B: p = {stats.shapiro(store_b).pvalue:.3f}")

res = stats.ttest_ind(store_a, store_b, equal_var=False)
ci = res.confidence_interval(0.95)
print(f"\nWelch の t 検定: t = {res.statistic:.3f}, p 値 = {res.pvalue:.5f}")
print(f"平均の差 (A - B) の 95% CI: [{ci.low:.2f}, {ci.high:.2f}]")
print(f"Cohen's d = {cohens_d(store_a, store_b):.2f}")
print(f"（参考）Mann-Whitney の U 検定: p 値 = {stats.mannwhitneyu(store_a, store_b).pvalue:.5f}")

plt.figure(figsize=(4.5, 3.3))
plt.boxplot([store_a, store_b])
plt.xticks([1, 2], ["店舗 A", "店舗 B"])
plt.ylabel("1 日の売上 (万円)")
plt.show()

**解釈**: 店舗 B の平均は A より約 9 万円高く、p 値は 0.001 未満、差の信頼区間も 0 を含みません。効果量 d ≈ 1.9 と非常に大きな差です。店舗 B の方が売上が多いと言えます。ただし、曜日や立地など他の要因の影響は考慮されていないことに注意が必要です。

### 問題 2: 減量プログラムの効果

10 人が 3 か月の減量プログラムに参加し、前後の体重（kg）を測定しました。プログラムに効果はあったでしょうか。

- 前: `82.5, 76.0, 90.2, 68.4, 79.9, 85.1, 72.3, 88.0, 74.6, 81.2`
- 後: `80.1, 75.2, 87.9, 68.0, 77.5, 83.0, 72.5, 85.4, 73.9, 79.8`

In [ ]:
# 問題 2: ここにコードを書いてください

**解答例**

同じ人の前後の測定なので **対応のある t 検定**（`stats.ttest_rel`）を使います。「減った」ことを示したいので片側検定でも構いませんが、ここでは両側で検定します。

In [ ]:
w_before = np.array([82.5, 76.0, 90.2, 68.4, 79.9, 85.1, 72.3, 88.0, 74.6, 81.2])
w_after = np.array([80.1, 75.2, 87.9, 68.0, 77.5, 83.0, 72.5, 85.4, 73.9, 79.8])
w_diff = w_after - w_before

print(f"体重の変化 (後 - 前): {w_diff.round(1).tolist()}")
print(f"変化の平均 {w_diff.mean():+.2f} kg, SD {w_diff.std(ddof=1):.2f} kg, 減った人数 {(w_diff < 0).sum()} / {len(w_diff)}")

res = stats.ttest_rel(w_after, w_before)
ci = stats.t.interval(0.95, df=len(w_diff) - 1, loc=w_diff.mean(), scale=stats.sem(w_diff))
print(f"対応のある t 検定: t = {res.statistic:.3f}, p 値 = {res.pvalue:.5f}")
print(f"変化の平均の 95% CI: [{ci[0]:.2f}, {ci[1]:.2f}] kg")
print(f"効果量 (差の平均 / 差の SD) = {w_diff.mean() / w_diff.std(ddof=1):.2f}")
print(f"（参考）Wilcoxon の符号順位検定: p 値 = {stats.wilcoxon(w_after, w_before).pvalue:.4f}")

**解釈**: 体重は平均 1.5 kg 減少し、p 値は 0.01 未満、信頼区間も 0 を含みません。統計的には有意な減少です。ただし「1.5 kg（約 2%）」という減少量が実用的に意味のある大きさかは別の問題です。また、対照群がないので「プログラムのおかげ」とは断定できません（季節変動や測定時期の影響など）。

### 問題 3: 曜日によって来店者数は違うか

ある店の 1 週間の曜日別来店者数（月〜日）は `95, 88, 92, 105, 130, 160, 150` でした。来店者数は曜日によらず均等と言えるでしょうか。

In [ ]:
# 問題 3: ここにコードを書いてください

**解答例**

7 つのカテゴリの度数が「均等」という理論比率に合っているかを調べるので、**カイ二乗の適合度検定**（`stats.chisquare`）を使います。

In [ ]:
visitors_by_day = np.array([95, 88, 92, 105, 130, 160, 150])
days = ["月", "火", "水", "木", "金", "土", "日"]
expected_by_day = np.full(7, visitors_by_day.sum() / 7)

res = stats.chisquare(f_obs=visitors_by_day, f_exp=expected_by_day)
print(f"合計 {visitors_by_day.sum()} 人, 均等なら 1 日あたり {expected_by_day[0]:.1f} 人")
print(f"χ² = {res.statistic:.2f}, 自由度 = 6, p 値 = {res.pvalue:.2g}")

contribution = (visitors_by_day - expected_by_day) ** 2 / expected_by_day
print("\n各曜日の χ² への寄与（大きい曜日ほど期待からずれている）:")
print(pd.Series(contribution, index=days).round(2).to_string())

plt.figure(figsize=(5.5, 3.3))
plt.bar(days, visitors_by_day, color="steelblue")
plt.axhline(expected_by_day[0], color="red", linestyle="--", label="均等な場合")
plt.ylabel("来店者数")
plt.legend()
plt.show()

**解釈**: p 値は極めて小さく、「曜日によらず均等」という仮説は棄却されます。χ² への寄与を見ると土曜・日曜・金曜が大きく、週末に来店が集中していることが分かります。1 週間分のデータなので、複数週のデータで再確認するとより確かです。

### 問題 4: 気温とアイスコーヒーの販売数

10 日間の最高気温（°C）とアイスコーヒーの販売数（杯）を記録しました。

- 気温: `22, 25, 27, 29, 30, 31, 33, 34, 35, 36`
- 販売数: `60, 72, 85, 95, 98, 110, 120, 128, 135, 140`

(1) 相関係数を求めて関係の強さを評価し、(2) 回帰式を求めて気温 32°C の日の販売数を予測してください。

In [ ]:
# 問題 4: ここにコードを書いてください

**解答例**

2 つの量的変数の関係なので **ピアソンの相関係数** と **単回帰**（`stats.linregress`）を使います。

In [ ]:
temp_c = np.array([22, 25, 27, 29, 30, 31, 33, 34, 35, 36])
cups = np.array([60, 72, 85, 95, 98, 110, 120, 128, 135, 140])

r = stats.pearsonr(temp_c, cups)
print(f"相関係数 r = {r.statistic:.3f}, p 値 = {r.pvalue:.2g}")

reg = stats.linregress(temp_c, cups)
print(f"回帰式: 販売数 = {reg.intercept:.2f} + {reg.slope:.3f} × 気温,  R² = {reg.rvalue ** 2:.3f}")
print(f"気温が 1°C 上がるごとに約 {reg.slope:.1f} 杯増える（傾きの標準誤差 {reg.stderr:.2f}）")
print(f"気温 32°C の予測販売数: {reg.intercept + reg.slope * 32:.1f} 杯")

plt.figure(figsize=(5.5, 3.5))
plt.scatter(temp_c, cups, color="steelblue", label="観測値")
xs = np.linspace(20, 38, 50)
plt.plot(xs, reg.intercept + reg.slope * xs, color="red", label="回帰直線")
plt.scatter([32], [reg.intercept + reg.slope * 32], color="red", marker="x", s=80, label="32°C の予測")
plt.xlabel("最高気温 (°C)")
plt.ylabel("販売数 (杯)")
plt.legend()
plt.show()

**解釈**: r ≈ 0.99 の非常に強い正の相関があり、気温が 1°C 上がると販売数は約 5.9 杯増えます。32°C の予測は約 115 杯で、これはデータの範囲内（22〜36°C）の予測なので信頼できます。ただし n = 10 と少なく、40°C を超えるような範囲外の予測には使えません。

### 問題 5: 3 つの Web デザインの比較

3 種類のデザイン A, B, C のページについて、それぞれ 8 人の滞在時間（秒）を測りました。デザインによって滞在時間に差があるでしょうか。差があるなら、どのデザインとどのデザインの間に差があるでしょうか。

- A: `45, 52, 48, 60, 55, 50, 47, 58`
- B: `62, 70, 65, 58, 72, 68, 66, 74`
- C: `50, 55, 49, 61, 57, 53, 60, 52`

In [ ]:
# 問題 5: ここにコードを書いてください

**解答例**

3 群以上の平均の比較なので **一元配置分散分析**（`stats.f_oneway`）を行い、有意なら **Tukey の HSD** で多重比較します。

In [ ]:
design = {
    "A": np.array([45, 52, 48, 60, 55, 50, 47, 58]),
    "B": np.array([62, 70, 65, 58, 72, 68, 66, 74]),
    "C": np.array([50, 55, 49, 61, 57, 53, 60, 52]),
}
for name, v in design.items():
    print(f"デザイン {name}: 平均 {v.mean():.2f} 秒, SD {v.std(ddof=1):.2f}")

res = stats.f_oneway(*design.values())
print(f"\n分散分析: F = {res.statistic:.3f}, p 値 = {res.pvalue:.2g}")

all_values = np.concatenate(list(design.values()))
all_labels = np.repeat(list(design.keys()), 8)
print("\nTukey の HSD:")
print(pairwise_tukeyhsd(endog=all_values, groups=all_labels, alpha=0.05))
print(f"（参考）Kruskal-Wallis 検定: p 値 = {stats.kruskal(*design.values()).pvalue:.2g}")

**解釈**: 分散分析の p 値は 0.001 未満で、デザインによって滞在時間に差があります。Tukey の HSD では B と A、B と C の間に有意差があり（B の方が約 12〜15 秒長い）、A と C の間には有意差がありません。デザイン B が滞在時間を伸ばすと言えそうです。

## 10. まとめと次のステップ

### どの場面でどの手法を使うか

| 目的 | データの種類 | 手法 | 関数 |
|---|---|---|---|
| データの要約 | 量的 | 平均・中央値・SD・四分位数 | `describe()`, `np.percentile` |
| 母平均の推定 | 量的 | t 分布による信頼区間 | `stats.t.interval` |
| 母比率の推定 | 2 値 | Wilson 法の信頼区間 | `proportion_confint` |
| 公式のない統計量の区間推定 | 何でも | ブートストラップ | `rng.choice(..., replace=True)` |
| 1 群の平均 vs 基準値 | 量的 | 1 標本 t 検定 | `stats.ttest_1samp` |
| 独立 2 群の平均 | 量的（正規に近い） | Welch の t 検定 | `stats.ttest_ind(equal_var=False)` |
| 独立 2 群の分布 | 量的（歪み・順序） | Mann-Whitney の U 検定 | `stats.mannwhitneyu` |
| 対応のある 2 回の測定 | 量的 | 対応のある t 検定 / Wilcoxon | `stats.ttest_rel`, `stats.wilcoxon` |
| 3 群以上の平均 | 量的 | 分散分析 + Tukey HSD | `stats.f_oneway`, `pairwise_tukeyhsd` |
| 3 群以上の分布 | 量的（歪み） | Kruskal-Wallis 検定 | `stats.kruskal` |
| 比率 vs 基準値 / 2 群の比率 | 2 値 | 比率の z 検定・二項検定 | `proportions_ztest`, `stats.binomtest` |
| 度数が理論比率に合うか | カテゴリ | 適合度検定 | `stats.chisquare` |
| 2 つのカテゴリ変数の関連 | カテゴリ | 独立性の検定 | `stats.chi2_contingency` |
| 正規性の確認 | 量的 | Shapiro-Wilk + Q-Q プロット | `stats.shapiro`, `stats.probplot` |
| 2 変数の直線的な関係 | 量的 | ピアソン相関 | `stats.pearsonr` |
| 単調な関係・順位 | 量的 / 順序 | スピアマン相関 | `stats.spearmanr` |
| 1 つの説明変数で予測 | 量的 | 単回帰 | `stats.linregress` |
| 複数の説明変数で予測 | 量的 | 重回帰 | `statsmodels ols` |
| 2 値の結果を予測 | 2 値 | ロジスティック回帰 | `statsmodels logit` |
| 分布を仮定しない検定 | 何でも | 並べ替え検定 | 自作 / `stats.permutation_test` |
| 時系列の傾向・季節性 | 時系列 | 移動平均・前年同月比・季節分解 | `rolling`, `shift`, `seasonal_decompose` |

### 分析するときの心得

1. **まずグラフを描く**。ヒストグラム、箱ひげ図、散布図で分布・外れ値・グループ構造を確認する。
2. **p 値だけで判断しない**。効果量と信頼区間を必ず併記する。
3. **検定の前提を確認する**。正規性、等分散、独立性、対応の有無。
4. **仮説と分析計画は事前に決める**。データを見てから検定や片側・両側を変えない。
5. **相関は因果ではない**。交絡や逆因果を疑い、可能なら無作為化実験を行う。
6. **サンプルサイズを意識する**。小さすぎれば検出できず、大きすぎれば何でも有意になる。

### 次のステップ

- [scipy.stats のドキュメント](https://docs.scipy.org/doc/scipy/reference/stats.html) — 確率分布と検定の一覧
- [statsmodels のドキュメント](https://www.statsmodels.org/stable/index.html) — 回帰分析、分散分析、時系列分析（[日本語の使い方例](https://www.statsmodels.org/stable/examples/index.html) も豊富）
- [pandas ユーザーガイド](https://pandas.pydata.org/docs/user_guide/index.html) — groupby、時系列、欠損値の扱い
- [matplotlib チュートリアル](https://matplotlib.org/stable/tutorials/index.html) — グラフの細かい調整
- 発展的な話題: ベイズ統計（`pymc`）、機械学習（`scikit-learn`）、因果推論、時系列予測（ARIMA、状態空間モデル）

このノートブックのコードは自由に書き換えて試してください。乱数のシードや n を変えて、結果がどう変わるかを観察することが理解への近道です。